<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/04_Statistical_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# NOTEBOOK 04 — STATISTICAL BASELINES
# SPP-GAN: A Privacy-Preserving Statistical–Machine Learning Framework for
# High-Fidelity Synthetic Tabular Data Generation
# ==================================================================================================
#
# PURPOSE
# -------
# Establish reproducible statistical baseline generators using the authoritative
# native TRAINING data persisted by Notebook 02.
#
# BASELINES
# ---------
# 1. Independent Marginal Sampling
# 2. Gaussian Copula
#
# DATA POLICY
# -----------
# - TRAIN split only is used for fitting.
# - VALIDATION and TEST are never used for fitting.
# - Notebook 02 preprocessing is NOT refitted.
# - Raw data are NOT reloaded.
#
# SCHEMA POLICY
# -------------
# Native generative schema:
#     preprocessing features + target
#
# Statistical model input:
#     generative columns
#
# Target:
#     generated as part of the synthetic generative table
#     NEVER used as a predictor
#
# Identifier / provenance:
#     excluded from synthetic generation
#
# RAM POLICY
# ----------
# - One dataset at a time.
# - One baseline at a time.
# - No dataset concatenation.
# - Persist artifacts immediately.
# - Explicit garbage collection.
#
# ==================================================================================================

print("=" * 100)
print("NOTEBOOK 04 — STATISTICAL BASELINES")
print("=" * 100)

NOTEBOOK_ID = "04"
NOTEBOOK_NAME = "Statistical Baselines"
NOTEBOOK_VERSION = "2.0"

print(f"Notebook       : {NOTEBOOK_ID} — {NOTEBOOK_NAME}")
print(f"Version        : {NOTEBOOK_VERSION}")
print("Purpose        : Reproducible statistical synthetic-data baselines")
print("Data policy    : TRAIN ONLY")
print("Schema policy  : Notebook 02 native generative schema")
print("RAM policy     : One dataset / one baseline at a time")

NOTEBOOK 04 — STATISTICAL BASELINES
Notebook       : 04 — Statistical Baselines
Version        : 2.0
Purpose        : Reproducible statistical synthetic-data baselines
Data policy    : TRAIN ONLY
Schema policy  : Notebook 02 native generative schema
RAM policy     : One dataset / one baseline at a time


In [10]:
# ==================================================================================================
# NOTEBOOK 04 — STATISTICAL BASELINES
# SECTION 02 — LOAD CONFIGURATION
# ==================================================================================================
#
# PURPOSE
# -------
# Load and validate the authoritative configuration persisted by Notebook 00.
#
# IMPORTANT RESEARCH POLICY
# -------------------------
# • Notebook 00 is the authoritative configuration source.
# • Notebook 00 artifacts are JSON, not YAML.
# • Notebook 00 is FROZEN and must not be modified by Notebook 04.
# • Notebook 04 must not silently create or replace configuration values.
# • Statistical baseline definitions are specified separately in Section 05.
# • Only TRAIN data may be used for fitting statistical baselines.
# • Validation and TEST data remain reserved for downstream evaluation.
#
# AUTHORITATIVE NOTEBOOK 00 ARTIFACT ROOT
# ---------------------------------------
# /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/
#
# EXPECTED ARTIFACTS
# ------------------
# config/
#   experiment_config.json
#   dataset_registry.json
#   model_registry.json
#   evaluation_config.json
#   privacy_config.json
#   sppgan_config.json
#
# environment/
#   environment.json
#
# manifest/
#   notebook_00_manifest.json
#   configuration_fingerprint.json
#
# ==================================================================================================

print("=" * 100)
print("SECTION 02 — LOAD CONFIGURATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

from pathlib import Path
import json
import hashlib
import gc


# --------------------------------------------------------------------------------------------------
# 2. Verify Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"

if not DRIVE_ROOT.exists():
    raise RuntimeError(
        "Google Drive is not mounted.\n"
        "Mount Google Drive before executing Notebook 04."
    )

if not MYDRIVE_ROOT.exists():
    raise RuntimeError(
        f"MyDrive directory not found:\n{MYDRIVE_ROOT}"
    )

print(f"✓ Google Drive verified : {DRIVE_ROOT}")
print(f"✓ MyDrive verified      : {MYDRIVE_ROOT}")


# --------------------------------------------------------------------------------------------------
# 3. Define Canonical Project Root
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Canonical project root does not exist:\n{PROJECT_ROOT}"
    )

print(f"✓ Project root verified : {PROJECT_ROOT}")


# --------------------------------------------------------------------------------------------------
# 4. Define Canonical Notebook 00 Artifact Root
# --------------------------------------------------------------------------------------------------

NB00_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_00"
)

NB00_CONFIG_ROOT = NB00_ROOT / "config"
NB00_ENV_ROOT = NB00_ROOT / "environment"
NB00_MANIFEST_ROOT = NB00_ROOT / "manifest"


print()
print("-" * 100)
print("NOTEBOOK 00 ARTIFACT ROOTS")
print("-" * 100)

print(f"✓ Notebook 00 root       : {NB00_ROOT}")
print(f"✓ Configuration root     : {NB00_CONFIG_ROOT}")
print(f"✓ Environment root       : {NB00_ENV_ROOT}")
print(f"✓ Manifest root          : {NB00_MANIFEST_ROOT}")


# --------------------------------------------------------------------------------------------------
# 5. Verify Notebook 00 Directory Structure
# --------------------------------------------------------------------------------------------------

REQUIRED_NB00_DIRECTORIES = {
    "Notebook 00 root": NB00_ROOT,
    "Configuration directory": NB00_CONFIG_ROOT,
    "Environment directory": NB00_ENV_ROOT,
    "Manifest directory": NB00_MANIFEST_ROOT,
}

for label, path in REQUIRED_NB00_DIRECTORIES.items():

    if not path.exists():
        raise RuntimeError(
            f"{label} is missing:\n{path}"
        )

    if not path.is_dir():
        raise RuntimeError(
            f"{label} exists but is not a directory:\n{path}"
        )

    print(f"✓ {label:<30}: {path}")


# --------------------------------------------------------------------------------------------------
# 6. Define Required Notebook 00 JSON Artifacts
# --------------------------------------------------------------------------------------------------

NB00_ARTIFACT_PATHS = {

    # Configuration artifacts
    "experiment_config":
        NB00_CONFIG_ROOT / "experiment_config.json",

    "dataset_registry":
        NB00_CONFIG_ROOT / "dataset_registry.json",

    "model_registry":
        NB00_CONFIG_ROOT / "model_registry.json",

    "evaluation_config":
        NB00_CONFIG_ROOT / "evaluation_config.json",

    "privacy_config":
        NB00_CONFIG_ROOT / "privacy_config.json",

    "sppgan_config":
        NB00_CONFIG_ROOT / "sppgan_config.json",

    # Environment artifact
    "environment":
        NB00_ENV_ROOT / "environment.json",

    # Manifest artifacts
    "manifest":
        NB00_MANIFEST_ROOT / "notebook_00_manifest.json",

    "configuration_fingerprint":
        NB00_MANIFEST_ROOT / "configuration_fingerprint.json",
}


# --------------------------------------------------------------------------------------------------
# 7. Verify Required Artifact Files
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 00 ARTIFACT DISCOVERY")
print("-" * 100)

for artifact_name, artifact_path in NB00_ARTIFACT_PATHS.items():

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Required Notebook 00 artifact is missing:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {artifact_path}"
        )

    if not artifact_path.is_file():
        raise RuntimeError(
            f"Notebook 00 artifact exists but is not a file:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {artifact_path}"
        )

    file_size = artifact_path.stat().st_size

    if file_size == 0:
        raise RuntimeError(
            f"Notebook 00 artifact is empty:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {artifact_path}"
        )

    print(
        f"✓ {artifact_name:<30} : "
        f"{artifact_path} "
        f"({file_size:,} bytes)"
    )


# --------------------------------------------------------------------------------------------------
# 8. JSON Loading Helper
# --------------------------------------------------------------------------------------------------

def load_json_artifact(path: Path, artifact_name: str):

    try:

        with path.open("r", encoding="utf-8") as file:
            data = json.load(file)

    except json.JSONDecodeError as exc:

        raise RuntimeError(
            f"Invalid JSON in Notebook 00 artifact:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {path}\n"
            f"  Error    : {exc}"
        ) from exc

    except Exception as exc:

        raise RuntimeError(
            f"Unable to read Notebook 00 artifact:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {path}\n"
            f"  Error    : {exc}"
        ) from exc

    return data


# --------------------------------------------------------------------------------------------------
# 9. Load All Notebook 00 Artifacts
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("LOADING NOTEBOOK 00 JSON ARTIFACTS")
print("-" * 100)

EXPERIMENT_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["experiment_config"],
    "experiment_config",
)

DATASET_REGISTRY = load_json_artifact(
    NB00_ARTIFACT_PATHS["dataset_registry"],
    "dataset_registry",
)

MODEL_REGISTRY = load_json_artifact(
    NB00_ARTIFACT_PATHS["model_registry"],
    "model_registry",
)

EVALUATION_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["evaluation_config"],
    "evaluation_config",
)

PRIVACY_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["privacy_config"],
    "privacy_config",
)

SPPGAN_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["sppgan_config"],
    "sppgan_config",
)

ENVIRONMENT = load_json_artifact(
    NB00_ARTIFACT_PATHS["environment"],
    "environment",
)

NOTEBOOK_00_MANIFEST = load_json_artifact(
    NB00_ARTIFACT_PATHS["manifest"],
    "manifest",
)

CONFIGURATION_FINGERPRINT = load_json_artifact(
    NB00_ARTIFACT_PATHS["configuration_fingerprint"],
    "configuration_fingerprint",
)


LOADED_ARTIFACTS = {
    "experiment_config": EXPERIMENT_CONFIG,
    "dataset_registry": DATASET_REGISTRY,
    "model_registry": MODEL_REGISTRY,
    "evaluation_config": EVALUATION_CONFIG,
    "privacy_config": PRIVACY_CONFIG,
    "sppgan_config": SPPGAN_CONFIG,
    "environment": ENVIRONMENT,
    "manifest": NOTEBOOK_00_MANIFEST,
    "configuration_fingerprint": CONFIGURATION_FINGERPRINT,
}


for artifact_name in LOADED_ARTIFACTS:

    print(f"✓ Loaded : {artifact_name}")


# --------------------------------------------------------------------------------------------------
# 10. Validate Experiment Configuration Structure
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("VALIDATING EXPERIMENT CONFIGURATION STRUCTURE")
print("-" * 100)

if not isinstance(EXPERIMENT_CONFIG, dict):
    raise RuntimeError(
        "Notebook 00 experiment_config.json must contain a dictionary/object."
    )

REQUIRED_EXPERIMENT_KEYS = {
    "advanced_analysis",
    "advanced_analysis_registry",
    "comparison_model_registry",
    "comparison_models",
    "data_split",
    "datasets",
    "evaluation",
    "evaluation_registry",
    "experimental_unit",
    "repetitions",
    "seed_policy",
}

MISSING_EXPERIMENT_KEYS = (
    REQUIRED_EXPERIMENT_KEYS
    - set(EXPERIMENT_CONFIG.keys())
)

if MISSING_EXPERIMENT_KEYS:

    raise RuntimeError(
        "Notebook 00 experiment configuration is missing required keys:\n"
        f"{sorted(MISSING_EXPERIMENT_KEYS)}"
    )

print("✓ Experiment configuration structure validated")


# --------------------------------------------------------------------------------------------------
# 11. Extract Authoritative Seed Policy
# --------------------------------------------------------------------------------------------------

SEED_POLICY = EXPERIMENT_CONFIG.get("seed_policy")

if not isinstance(SEED_POLICY, dict):

    raise RuntimeError(
        "Notebook 00 experiment_config.json contains an invalid "
        "'seed_policy' structure."
    )


MASTER_SEED = SEED_POLICY.get("master_seed")

if MASTER_SEED is None:

    raise RuntimeError(
        "Notebook 00 experiment configuration does not contain the "
        "authoritative master seed at 'seed_policy.master_seed'."
    )

MASTER_SEED = int(MASTER_SEED)


DETERMINISTIC = bool(
    SEED_POLICY.get("deterministic", True)
)


REPETITIONS = int(
    SEED_POLICY.get(
        "repetitions",
        EXPERIMENT_CONFIG.get("repetitions", 5),
    )
)


REPETITION_SEED_OFFSET = int(
    SEED_POLICY.get(
        "repetition_seed_offset",
        1000,
    )
)


REPETITION_SEEDS = SEED_POLICY.get(
    "repetition_seeds",
    {},
)


if not isinstance(REPETITION_SEEDS, dict):

    raise RuntimeError(
        "Notebook 00 'seed_policy.repetition_seeds' "
        "must be a dictionary."
    )


print("✓ Authoritative seed policy loaded")
print(f"  Master seed            : {MASTER_SEED}")
print(f"  Deterministic          : {DETERMINISTIC}")
print(f"  Repetitions            : {REPETITIONS}")
print(f"  Repetition seed offset : {REPETITION_SEED_OFFSET}")
print(f"  Repetition seeds       : {REPETITION_SEEDS}")


# --------------------------------------------------------------------------------------------------
# 12. Validate Repetition Seed Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_REPETITION_KEYS = {
    str(i)
    for i in range(1, REPETITIONS + 1)
}

ACTUAL_REPETITION_KEYS = set(
    str(key)
    for key in REPETITION_SEEDS.keys()
)

if ACTUAL_REPETITION_KEYS != EXPECTED_REPETITION_KEYS:

    raise RuntimeError(
        "Notebook 00 repetition seed registry is inconsistent.\n"
        f"Expected : {sorted(EXPECTED_REPETITION_KEYS)}\n"
        f"Found    : {sorted(ACTUAL_REPETITION_KEYS)}"
    )


for repetition in range(1, REPETITIONS + 1):

    repetition_key = str(repetition)

    repetition_seed = int(
        REPETITION_SEEDS[repetition_key]
    )

    expected_seed = (
        MASTER_SEED
        + REPETITION_SEED_OFFSET
        + repetition
    )

    if repetition_seed != expected_seed:

        raise RuntimeError(
            "Notebook 00 repetition seed does not match the "
            "authoritative seed policy.\n"
            f"  Repetition : {repetition}\n"
            f"  Expected   : {expected_seed}\n"
            f"  Found      : {repetition_seed}"
        )


print(
    f"✓ Repetition seed registry validated "
    f"({REPETITIONS} repetitions)"
)


# --------------------------------------------------------------------------------------------------
# 13. Validate Dataset Registry
# --------------------------------------------------------------------------------------------------

if not isinstance(DATASET_REGISTRY, dict):

    raise RuntimeError(
        "Notebook 00 dataset_registry.json must contain a dictionary/object."
    )


EXPECTED_DATASETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


def extract_dataset_entries(registry):

    """
    Resolve the dataset registry into a dictionary keyed by dataset ID.

    Notebook 00 is authoritative. This helper only resolves the persisted
    structure and does not create or modify configuration.
    """

    if all(
        dataset_id in registry
        for dataset_id in EXPECTED_DATASETS
    ):
        return registry

    for key in (
        "datasets",
        "registry",
        "dataset_registry",
    ):

        candidate = registry.get(key)

        if isinstance(candidate, dict):

            if all(
                dataset_id in candidate
                for dataset_id in EXPECTED_DATASETS
            ):
                return candidate

    raise RuntimeError(
        "Unable to resolve the authoritative dataset registry structure "
        "from Notebook 00."
    )


DATASET_ENTRIES = extract_dataset_entries(
    DATASET_REGISTRY
)


for dataset_id, expected_target in EXPECTED_DATASETS.items():

    if dataset_id not in DATASET_ENTRIES:

        raise RuntimeError(
            f"Required dataset '{dataset_id}' is missing "
            "from Notebook 00 dataset registry."
        )

    dataset_entry = DATASET_ENTRIES[dataset_id]

    if not isinstance(dataset_entry, dict):

        raise RuntimeError(
            f"Dataset registry entry for '{dataset_id}' "
            "must be a dictionary."
        )

    target_candidates = [
        dataset_entry.get("target"),
        dataset_entry.get("target_column"),
        dataset_entry.get("label"),
    ]

    target_value = next(
        (
            value
            for value in target_candidates
            if value is not None
        ),
        None,
    )

    if target_value != expected_target:

        raise RuntimeError(
            f"Dataset target mismatch for '{dataset_id}'.\n"
            f"Expected : {expected_target}\n"
            f"Found    : {target_value}"
        )


print("✓ Dataset registry validated")
print(f"  Registered datasets : {sorted(EXPECTED_DATASETS.keys())}")
print(f"  Target mapping      : {EXPECTED_DATASETS}")


# --------------------------------------------------------------------------------------------------
# 14. Validate Experiment Data-Split Policy
# --------------------------------------------------------------------------------------------------

DATA_SPLIT_CONFIG = EXPERIMENT_CONFIG.get("data_split", {})

if not isinstance(DATA_SPLIT_CONFIG, dict):

    raise RuntimeError(
        "Notebook 00 'data_split' must be a dictionary."
    )


EXPECTED_FIT_SPLIT = "train"

if DATA_SPLIT_CONFIG.get("fit_split") != EXPECTED_FIT_SPLIT:

    raise RuntimeError(
        "Notebook 04 requires TRAIN-only fitting.\n"
        f"Expected fit_split : {EXPECTED_FIT_SPLIT}\n"
        f"Found              : {DATA_SPLIT_CONFIG.get('fit_split')}"
    )


EXPECTED_SPLITS = {
    "train",
    "validation",
    "test",
}

if not DATA_SPLIT_CONFIG.get("enabled", False):

    raise RuntimeError(
        "Notebook 00 data splitting is disabled. "
        "Notebook 04 requires the authoritative train/validation/test split."
    )


print("✓ Data-split policy validated")
print(f"  Fit split       : {DATA_SPLIT_CONFIG.get('fit_split')}")
print(f"  Train fraction  : {DATA_SPLIT_CONFIG.get('train_fraction')}")
print(f"  Validation      : {DATA_SPLIT_CONFIG.get('validation_fraction')}")
print(f"  Test fraction   : {DATA_SPLIT_CONFIG.get('test_fraction')}")
print(f"  Required splits : {sorted(EXPECTED_SPLITS)}")


# --------------------------------------------------------------------------------------------------
# 15. Validate Comparison Model Registry
# --------------------------------------------------------------------------------------------------

if not isinstance(MODEL_REGISTRY, dict):

    raise RuntimeError(
        "Notebook 00 model_registry.json must contain a dictionary/object."
    )


EXPECTED_COMPARISON_MODELS = {
    "statistical",
    "tvae",
    "ctgan",
    "dp_ctgan",
    "spp_gan",
}


MODEL_REGISTRY_KEYS = set(
    MODEL_REGISTRY.keys()
)

MODEL_REGISTRY_CANDIDATE = MODEL_REGISTRY

if not EXPECTED_COMPARISON_MODELS.issubset(
    MODEL_REGISTRY_KEYS
):

    for key in (
        "models",
        "comparison_models",
        "comparison_model_registry",
        "registry",
    ):

        candidate = MODEL_REGISTRY.get(key)

        if isinstance(candidate, dict):

            if EXPECTED_COMPARISON_MODELS.issubset(
                set(candidate.keys())
            ):
                MODEL_REGISTRY_CANDIDATE = candidate
                break


MODEL_REGISTRY_ENTRIES = MODEL_REGISTRY_CANDIDATE


MISSING_MODELS = (
    EXPECTED_COMPARISON_MODELS
    - set(MODEL_REGISTRY_ENTRIES.keys())
)

if MISSING_MODELS:

    raise RuntimeError(
        "Notebook 00 model registry is missing expected "
        f"comparison models: {sorted(MISSING_MODELS)}"
    )


print("✓ Comparison model registry validated")
print(
    f"  Registered comparison models : "
    f"{sorted(EXPECTED_COMPARISON_MODELS)}"
)


# --------------------------------------------------------------------------------------------------
# 16. Validate Notebook 04 Baseline Identifiers
# --------------------------------------------------------------------------------------------------
#
# These identifiers define the exact statistical baselines used by Notebook 04.
# Their mathematical definitions belong to Section 05.
#
# --------------------------------------------------------------------------------------------------

STATISTICAL_BASELINE_IDS = [
    "independent_marginal",
    "gaussian_copula",
]

if len(STATISTICAL_BASELINE_IDS) != len(
    set(STATISTICAL_BASELINE_IDS)
):

    raise RuntimeError(
        "Notebook 04 statistical baseline identifiers contain duplicates."
    )


print("✓ Statistical baseline identifiers validated")
print(
    f"  Baselines : {STATISTICAL_BASELINE_IDS}"
)


# --------------------------------------------------------------------------------------------------
# 17. Validate Experimental Unit
# --------------------------------------------------------------------------------------------------

EXPERIMENTAL_UNIT = EXPERIMENT_CONFIG.get(
    "experimental_unit",
    {}
)

if not isinstance(EXPERIMENTAL_UNIT, dict):

    raise RuntimeError(
        "Notebook 00 'experimental_unit' must be a dictionary."
    )


PRIMARY_EXPERIMENTAL_UNIT = EXPERIMENTAL_UNIT.get(
    "primary_unit"
)

EXPECTED_PRIMARY_UNIT = (
    "dataset × repetition × method"
)

if PRIMARY_EXPERIMENTAL_UNIT != EXPECTED_PRIMARY_UNIT:

    raise RuntimeError(
        "Unexpected primary experimental unit.\n"
        f"Expected : {EXPECTED_PRIMARY_UNIT}\n"
        f"Found    : {PRIMARY_EXPERIMENTAL_UNIT}"
    )


print("✓ Experimental unit validated")
print(f"  Primary unit : {PRIMARY_EXPERIMENTAL_UNIT}")


# --------------------------------------------------------------------------------------------------
# 18. Validate Notebook 00 Manifest Identity
# --------------------------------------------------------------------------------------------------

if not isinstance(
    NOTEBOOK_00_MANIFEST,
    dict
):

    raise RuntimeError(
        "Notebook 00 manifest must contain a dictionary/object."
    )


MANIFEST_VERSION = NOTEBOOK_00_MANIFEST.get(
    "manifest_version"
)

if MANIFEST_VERSION != "2.0":

    raise RuntimeError(
        "Unexpected Notebook 00 manifest version.\n"
        f"Expected : 2.0\n"
        f"Found    : {MANIFEST_VERSION}"
    )


PROJECT_INFO = NOTEBOOK_00_MANIFEST.get(
    "project",
    {}
)

NOTEBOOK_INFO = NOTEBOOK_00_MANIFEST.get(
    "notebook",
    {}
)


if PROJECT_INFO.get("name") != "SPP-GAN Research Project":

    raise RuntimeError(
        "Notebook 00 manifest project identity mismatch.\n"
        f"Found : {PROJECT_INFO.get('name')}"
    )


if PROJECT_INFO.get("version") != "1.0":

    raise RuntimeError(
        "Notebook 00 manifest project version mismatch.\n"
        f"Expected : 1.0\n"
        f"Found    : {PROJECT_INFO.get('version')}"
    )


if str(NOTEBOOK_INFO.get("id")) != "00":

    raise RuntimeError(
        "Notebook 00 manifest notebook ID mismatch.\n"
        f"Expected : 00\n"
        f"Found    : {NOTEBOOK_INFO.get('id')}"
    )


if NOTEBOOK_INFO.get("name") != (
    "Environment, Configuration & Reproducibility"
):

    raise RuntimeError(
        "Notebook 00 manifest notebook name mismatch.\n"
        f"Found : {NOTEBOOK_INFO.get('name')}"
    )


print("✓ Notebook 00 manifest identity validated")
print(f"  Project  : {PROJECT_INFO.get('name')} v{PROJECT_INFO.get('version')}")
print(f"  Notebook : {NOTEBOOK_INFO.get('id')} — {NOTEBOOK_INFO.get('name')}")
print(f"  Manifest : v{MANIFEST_VERSION}")


# --------------------------------------------------------------------------------------------------
# 19. Validate Notebook 00 Final Integrity Chain
# --------------------------------------------------------------------------------------------------

INTEGRITY = NOTEBOOK_00_MANIFEST.get(
    "integrity",
    {}
)

REQUIRED_INTEGRITY_FLAGS = [
    "section_17_pass",
    "section_18_pass",
    "section_19_pass",
    "section_20_pass",
    "section_21_pass",
]


for flag in REQUIRED_INTEGRITY_FLAGS:

    if INTEGRITY.get(flag) is not True:

        raise RuntimeError(
            f"Notebook 00 final integrity flag failed: {flag}"
        )


print("✓ Notebook 00 final integrity chain validated")
print("  Sections 17–21 : PASS")


# --------------------------------------------------------------------------------------------------
# 20. Validate Configuration Fingerprint Artifact
# --------------------------------------------------------------------------------------------------

if not isinstance(
    CONFIGURATION_FINGERPRINT,
    dict
):

    raise RuntimeError(
        "Notebook 00 configuration_fingerprint.json "
        "must contain a dictionary/object."
    )


EXPECTED_CONFIG_HASH = CONFIGURATION_FINGERPRINT.get(
    "hash"
)

FINGERPRINT_ALGORITHM = CONFIGURATION_FINGERPRINT.get(
    "algorithm"
)

FINGERPRINT_VERSION = CONFIGURATION_FINGERPRINT.get(
    "fingerprint_version"
)


if not EXPECTED_CONFIG_HASH:

    raise RuntimeError(
        "Notebook 00 configuration fingerprint does not "
        "contain the authoritative 'hash' field."
    )


if FINGERPRINT_ALGORITHM != "SHA256":

    raise RuntimeError(
        "Unexpected configuration fingerprint algorithm.\n"
        f"Expected : SHA256\n"
        f"Found    : {FINGERPRINT_ALGORITHM}"
    )


if FINGERPRINT_VERSION != "1.0":

    raise RuntimeError(
        "Unexpected configuration fingerprint version.\n"
        f"Expected : 1.0\n"
        f"Found    : {FINGERPRINT_VERSION}"
    )


print("✓ Configuration fingerprint metadata validated")
print(f"  Algorithm : {FINGERPRINT_ALGORITHM}")
print(f"  Version   : {FINGERPRINT_VERSION}")
print(f"  Hash      : {EXPECTED_CONFIG_HASH}")


# --------------------------------------------------------------------------------------------------
# 21. Validate Train-Only Statistical Baseline Policy
# --------------------------------------------------------------------------------------------------

BASELINE_POLICY = {
    "fit_split": "train",
    "validation_used_for_fitting": False,
    "test_used_for_fitting": False,
    "target_used_as_predictor": False,
    "identifier_features_used": False,
    "provenance_features_used": False,
    "encoded_feature_matrix_used": False,
}


if BASELINE_POLICY["fit_split"] != "train":

    raise RuntimeError(
        "Statistical baselines must be fitted on TRAIN only."
    )


if BASELINE_POLICY["validation_used_for_fitting"]:

    raise RuntimeError(
        "Validation data must never be used for baseline fitting."
    )


if BASELINE_POLICY["test_used_for_fitting"]:

    raise RuntimeError(
        "Test data must never be used for baseline fitting."
    )


if BASELINE_POLICY["target_used_as_predictor"]:

    raise RuntimeError(
        "The target must not be used as a predictor."
    )


if BASELINE_POLICY["identifier_features_used"]:

    raise RuntimeError(
        "Explicit identifiers must not be used as generative features."
    )


if BASELINE_POLICY["provenance_features_used"]:

    raise RuntimeError(
        "Provenance fields must not be used as generative features."
    )


if BASELINE_POLICY["encoded_feature_matrix_used"]:

    raise RuntimeError(
        "Notebook 04 statistical baselines must operate on the "
        "canonical native generative representation, not the "
        "Notebook 02 encoded feature matrix."
    )


print("✓ Train-only baseline policy validated")
print("  Fitting split       : TRAIN")
print("  Validation fitting  : DISABLED")
print("  Test fitting        : DISABLED")
print("  Target as predictor : DISABLED")
print("  Explicit IDs        : EXCLUDED")
print("  Provenance          : EXCLUDED")
print("  Encoded matrix      : EXCLUDED")


# --------------------------------------------------------------------------------------------------
# 22. Freeze Configuration Snapshot for Notebook 04
# --------------------------------------------------------------------------------------------------

NB04_CONFIGURATION = {
    "project_root": str(PROJECT_ROOT),

    "notebook_00_root": str(NB00_ROOT),

    "master_seed": MASTER_SEED,

    "deterministic": DETERMINISTIC,

    "repetitions": REPETITIONS,

    "repetition_seed_offset": REPETITION_SEED_OFFSET,

    "repetition_seeds": {
        str(key): int(value)
        for key, value in REPETITION_SEEDS.items()
    },

    "datasets": EXPECTED_DATASETS.copy(),

    "expected_splits": sorted(EXPECTED_SPLITS),

    "fit_split": EXPECTED_FIT_SPLIT,

    "statistical_baselines": STATISTICAL_BASELINE_IDS.copy(),

    "primary_experimental_unit": PRIMARY_EXPERIMENTAL_UNIT,

    "baseline_policy": BASELINE_POLICY.copy(),

    "notebook_00_manifest_version": MANIFEST_VERSION,

    "notebook_00_configuration_hash": EXPECTED_CONFIG_HASH,
}


# --------------------------------------------------------------------------------------------------
# 23. Create Deterministic Configuration Digest for Notebook 04
# --------------------------------------------------------------------------------------------------

NB04_CONFIGURATION_SERIALIZED = json.dumps(
    NB04_CONFIGURATION,
    sort_keys=True,
    separators=(",", ":"),
)

NB04_CONFIGURATION_HASH = hashlib.sha256(
    NB04_CONFIGURATION_SERIALIZED.encode("utf-8")
).hexdigest()


print()
print("-" * 100)
print("NOTEBOOK 04 CONFIGURATION SNAPSHOT")
print("-" * 100)

print(f"✓ Master seed              : {MASTER_SEED}")
print(f"✓ Repetitions              : {REPETITIONS}")
print(f"✓ Fit split                : {EXPECTED_FIT_SPLIT}")
print(f"✓ Statistical baselines    : {STATISTICAL_BASELINE_IDS}")
print(f"✓ Datasets                 : {list(EXPECTED_DATASETS.keys())}")
print(f"✓ Experimental unit        : {PRIMARY_EXPERIMENTAL_UNIT}")
print(f"✓ Notebook 00 config hash  : {EXPECTED_CONFIG_HASH}")
print(f"✓ Notebook 04 config hash  : {NB04_CONFIGURATION_HASH}")


# --------------------------------------------------------------------------------------------------
# 24. Final Section 02 Verification
# --------------------------------------------------------------------------------------------------

SECTION_02_CHECKS = {

    "google_drive_verified":
        DRIVE_ROOT.exists(),

    "project_root_verified":
        PROJECT_ROOT.exists(),

    "notebook_00_root_verified":
        NB00_ROOT.exists(),

    "all_required_artifacts_present":
        all(
            path.exists() and path.is_file()
            for path in NB00_ARTIFACT_PATHS.values()
        ),

    "experiment_config_loaded":
        isinstance(EXPERIMENT_CONFIG, dict),

    "dataset_registry_loaded":
        isinstance(DATASET_REGISTRY, dict),

    "model_registry_loaded":
        isinstance(MODEL_REGISTRY, dict),

    "evaluation_config_loaded":
        isinstance(EVALUATION_CONFIG, dict),

    "privacy_config_loaded":
        isinstance(PRIVACY_CONFIG, dict),

    "sppgan_config_loaded":
        isinstance(SPPGAN_CONFIG, dict),

    "environment_loaded":
        isinstance(ENVIRONMENT, dict),

    "manifest_loaded":
        isinstance(NOTEBOOK_00_MANIFEST, dict),

    "configuration_fingerprint_loaded":
        isinstance(CONFIGURATION_FINGERPRINT, dict),

    "master_seed_verified":
        MASTER_SEED == 2025,

    "deterministic_policy_verified":
        DETERMINISTIC is True,

    "repetition_count_verified":
        REPETITIONS == 5,

    "repetition_seed_registry_verified":
        ACTUAL_REPETITION_KEYS == EXPECTED_REPETITION_KEYS,

    "train_only_fit_policy_verified":
        EXPECTED_FIT_SPLIT == "train",

    "dataset_registry_verified":
        set(EXPECTED_DATASETS.keys())
        == set(DATASET_ENTRIES.keys())
        or set(EXPECTED_DATASETS.keys()).issubset(
            set(DATASET_ENTRIES.keys())
        ),

    "experimental_unit_verified":
        PRIMARY_EXPERIMENTAL_UNIT
        == "dataset × repetition × method",

    "manifest_version_verified":
        MANIFEST_VERSION == "2.0",

    "manifest_project_verified":
        PROJECT_INFO.get("name")
        == "SPP-GAN Research Project",

    "manifest_notebook_verified":
        str(NOTEBOOK_INFO.get("id")) == "00",

    "manifest_integrity_verified":
        all(
            INTEGRITY.get(flag) is True
            for flag in REQUIRED_INTEGRITY_FLAGS
        ),

    "configuration_fingerprint_verified":
        bool(EXPECTED_CONFIG_HASH),

    "baseline_registry_verified":
        len(STATISTICAL_BASELINE_IDS) == 2,

    "independent_marginal_registered":
        "independent_marginal"
        in STATISTICAL_BASELINE_IDS,

    "gaussian_copula_registered":
        "gaussian_copula"
        in STATISTICAL_BASELINE_IDS,
}


FAILED_SECTION_02_CHECKS = [
    check_name
    for check_name, result in SECTION_02_CHECKS.items()
    if not result
]


print()
print("=" * 100)
print("SECTION 02 FINAL VERIFICATION")
print("=" * 100)

print(
    f"Total checks : {len(SECTION_02_CHECKS)}"
)

print(
    f"Passed       : "
    f"{sum(SECTION_02_CHECKS.values())}"
)

print(
    f"Failed       : "
    f"{len(FAILED_SECTION_02_CHECKS)}"
)


if FAILED_SECTION_02_CHECKS:

    print()
    print("FAILED CHECKS")
    print("-" * 100)

    for check_name in FAILED_SECTION_02_CHECKS:
        print(f"✗ {check_name}")

    raise RuntimeError(
        "Notebook 04 Section 02 verification FAILED."
    )


print()
print("✓ ALL SECTION 02 CHECKS PASSED")
print("✓ Notebook 00 authoritative configuration loaded successfully")
print("✓ Authoritative master seed : 2025")
print("✓ Repetition policy         : 5 repetitions")
print("✓ Train-only baseline policy verified")
print("✓ Notebook 00 integrity chain verified")
print("✓ Statistical baseline registry verified")
print()
print("SECTION 02 STATUS : PASS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 25. Memory Cleanup
# --------------------------------------------------------------------------------------------------

gc.collect()

SECTION 02 — LOAD CONFIGURATION
✓ Google Drive verified : /content/drive
✓ MyDrive verified      : /content/drive/MyDrive
✓ Project root verified : /content/drive/MyDrive/SPP_GAN_Research

----------------------------------------------------------------------------------------------------
NOTEBOOK 00 ARTIFACT ROOTS
----------------------------------------------------------------------------------------------------
✓ Notebook 00 root       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
✓ Configuration root     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config
✓ Environment root       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/environment
✓ Manifest root          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/manifest
✓ Notebook 00 root              : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
✓ Configuration directory       : /content/drive/MyDrive/SPP_G

24

In [26]:
# ==================================================================================================
# SECTION 03 — LOAD PROCESSED TRAINING DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 03 — LOAD PROCESSED TRAINING DATA")
print("=" * 100)

from pathlib import Path
import ast
import gc
import json
import hashlib
import pandas as pd
import numpy as np


# ==================================================================================================
# 1. VERIFY GOOGLE DRIVE AND PROJECT ROOT
# ==================================================================================================

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

if not DRIVE_ROOT.exists():
    raise RuntimeError(
        f"Google Drive not mounted: {DRIVE_ROOT}"
    )

if not MYDRIVE_ROOT.exists():
    raise RuntimeError(
        f"MyDrive not found: {MYDRIVE_ROOT}"
    )

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Project root not found: {PROJECT_ROOT}"
    )

print(f"✓ Google Drive     : {DRIVE_ROOT}")
print(f"✓ MyDrive          : {MYDRIVE_ROOT}")
print(f"✓ Project root     : {PROJECT_ROOT}")


# ==================================================================================================
# 2. CANONICAL NOTEBOOK 02 PATHS
# ==================================================================================================

NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

NB02_NATIVE_ROOT = (
    NB02_ROOT
    / "native"
)

NATIVE_MANIFEST_PATH = (
    NB02_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

if not NB02_ROOT.exists():
    raise RuntimeError(
        f"Notebook 02 root not found: {NB02_ROOT}"
    )

if not NB02_NATIVE_ROOT.exists():
    raise RuntimeError(
        f"Notebook 02 native directory not found: "
        f"{NB02_NATIVE_ROOT}"
    )

if not NATIVE_MANIFEST_PATH.exists():
    raise RuntimeError(
        f"Notebook 02 native manifest not found: "
        f"{NATIVE_MANIFEST_PATH}"
    )

print()
print(f"✓ Notebook 02 root : {NB02_ROOT}")
print(f"✓ Native data root : {NB02_NATIVE_ROOT}")
print(f"✓ Native manifest  : {NATIVE_MANIFEST_PATH}")


# ==================================================================================================
# 3. EXPECTED DATASET REGISTRY
# ==================================================================================================

EXPECTED_DATASETS = {
    "adult_income": {
        "target": "income",
        "identifiers": [],
    },
    "bank_marketing": {
        "target": "y",
        "identifiers": [],
    },
    "diabetes_130us": {
        "target": "readmitted",
        "identifiers": [
            "encounter_id",
            "patient_nbr",
        ],
    },
}

DATASET_IDS = list(
    EXPECTED_DATASETS.keys()
)

TARGET_COLUMNS = {
    dataset_id: EXPECTED_DATASETS[dataset_id]["target"]
    for dataset_id in DATASET_IDS
}

print()
print("Expected datasets:")

for dataset_id in DATASET_IDS:
    print(
        f"  • {dataset_id:<20} "
        f"target = "
        f"{EXPECTED_DATASETS[dataset_id]['target']}"
    )


# ==================================================================================================
# 4. HELPER FUNCTIONS
# ==================================================================================================

def normalize_scalar(value):
    """
    Normalize pandas / NumPy scalar values.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, np.generic):
        return value.item()

    return value


def parse_manifest_count(value, field_name):
    """
    Parse an integer count persisted in the Notebook 02 manifest.
    """

    value = normalize_scalar(value)

    if value is None:
        raise RuntimeError(
            f"{field_name} cannot be null."
        )

    if isinstance(
        value,
        bool,
    ):
        raise RuntimeError(
            f"{field_name} cannot be boolean."
        )

    try:
        return int(value)

    except Exception as exc:

        raise RuntimeError(
            f"Unable to parse {field_name}: "
            f"{value!r}"
        ) from exc


def parse_boolean(value):
    """
    Parse boolean-like manifest values.
    """

    value = normalize_scalar(value)

    if isinstance(value, bool):
        return value

    if isinstance(
        value,
        (int, np.integer),
    ):

        if value in (0, 1):
            return bool(value)

    if isinstance(value, str):

        normalized = (
            value
            .strip()
            .lower()
        )

        if normalized in {
            "true",
            "1",
            "yes",
            "y",
            "pass",
        }:
            return True

        if normalized in {
            "false",
            "0",
            "no",
            "n",
            "fail",
        }:
            return False

    raise RuntimeError(
        f"Unable to parse boolean manifest value: "
        f"{value!r}"
    )


def parse_identifier_columns(value):
    """
    Parse Notebook 02 identifier_columns.
    """

    value = normalize_scalar(value)

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return [
            str(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return [
            str(item)
            for item in value.tolist()
        ]

    if isinstance(value, str):

        text = value.strip()

        if not text:
            return []

        try:

            parsed = json.loads(text)

            if isinstance(
                parsed,
                list,
            ):

                return [
                    str(item)
                    for item in parsed
                ]

        except Exception:
            pass

        try:

            parsed = ast.literal_eval(text)

            if isinstance(
                parsed,
                (list, tuple),
            ):

                return [
                    str(item)
                    for item in parsed
                ]

        except Exception:
            pass

        if "," in text:

            return [
                item.strip()
                for item in text.split(",")
                if item.strip()
            ]

        return [text]

    raise RuntimeError(
        f"Unable to parse identifier_columns: "
        f"{value!r}"
    )


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    """
    Calculate SHA-256 without loading the complete file into RAM.
    """

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def resolve_manifest_path(
    relative_path,
    absolute_path,
):
    """
    Resolve canonical Notebook 02 native TRAIN file.
    """

    relative_path = normalize_scalar(
        relative_path
    )

    absolute_path = normalize_scalar(
        absolute_path
    )

    candidates = []

    if absolute_path:

        candidates.append(
            Path(
                str(absolute_path)
            )
        )

    if relative_path:

        relative = Path(
            str(relative_path)
        )

        candidates.extend([
            PROJECT_ROOT / relative,
            NB02_ROOT / relative,
            NB02_NATIVE_ROOT / relative,
        ])

    for candidate in candidates:

        if (
            candidate.exists()
            and candidate.is_file()
        ):

            return candidate

    raise FileNotFoundError(
        "Unable to resolve Notebook 02 native data file.\n"
        f"relative_path={relative_path!r}\n"
        f"absolute_path={absolute_path!r}"
    )


# ==================================================================================================
# 5. LOAD CANONICAL NOTEBOOK 02 NATIVE MANIFEST
# ==================================================================================================

print()
print("-" * 100)
print("LOAD NOTEBOOK 02 NATIVE MANIFEST")
print("-" * 100)

native_manifest = pd.read_csv(
    NATIVE_MANIFEST_PATH
)

print(
    "✓ Native manifest loaded"
)

print(
    f"  Rows    : {len(native_manifest):,}"
)

print(
    f"  Columns : {len(native_manifest.columns):,}"
)


# ==================================================================================================
# 6. VALIDATE EXACT MANIFEST SCHEMA
# ==================================================================================================

EXPECTED_MANIFEST_COLUMNS = [
    "dataset_id",
    "split",
    "relative_path",
    "absolute_path",
    "rows",
    "columns",
    "preprocessing_feature_columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "file_size_bytes",
    "sha256",
    "reload_validation",
    "status",
]

actual_manifest_columns = list(
    native_manifest.columns
)

missing_manifest_columns = [
    column
    for column in EXPECTED_MANIFEST_COLUMNS
    if column not in actual_manifest_columns
]

unexpected_manifest_columns = [
    column
    for column in actual_manifest_columns
    if column not in EXPECTED_MANIFEST_COLUMNS
]

if missing_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest is missing required columns:\n"
        + "\n".join(
            f"- {column}"
            for column in missing_manifest_columns
        )
    )

if unexpected_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest contains unexpected columns:\n"
        + "\n".join(
            f"- {column}"
            for column in unexpected_manifest_columns
        )
    )

if actual_manifest_columns != (
    EXPECTED_MANIFEST_COLUMNS
):

    raise RuntimeError(
        "Notebook 02 native manifest column order "
        "does not match the canonical schema."
    )

print(
    "✓ Exact required manifest schema verified"
)


# ==================================================================================================
# 7. VALIDATE MANIFEST ROW COUNT
# ==================================================================================================

EXPECTED_MANIFEST_ROWS = (
    len(DATASET_IDS) * 3
)

if len(native_manifest) != (
    EXPECTED_MANIFEST_ROWS
):

    raise RuntimeError(
        f"Expected {EXPECTED_MANIFEST_ROWS} "
        f"Notebook 02 native manifest records, "
        f"found {len(native_manifest)}."
    )

print(
    f"✓ Manifest row count verified: "
    f"{len(native_manifest)}"
)


# ==================================================================================================
# 8. VALIDATE DATASET COVERAGE
# ==================================================================================================

manifest_dataset_ids = sorted(
    native_manifest[
        "dataset_id"
    ]
    .astype(str)
    .unique()
)

if manifest_dataset_ids != (
    sorted(DATASET_IDS)
):

    raise RuntimeError(
        "Dataset coverage mismatch.\n"
        f"Expected: {sorted(DATASET_IDS)}\n"
        f"Found   : {manifest_dataset_ids}"
    )

print(
    "✓ Dataset coverage verified: "
    + ", ".join(DATASET_IDS)
)


# ==================================================================================================
# 9. VALIDATE SPLIT COVERAGE
# ==================================================================================================

EXPECTED_SPLITS = {
    "train",
    "validation",
    "test",
}

for dataset_id in DATASET_IDS:

    dataset_manifest = native_manifest[
        native_manifest[
            "dataset_id"
        ].astype(str)
        == dataset_id
    ]

    observed_splits = set(
        dataset_manifest[
            "split"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    if observed_splits != (
        EXPECTED_SPLITS
    ):

        raise RuntimeError(
            f"{dataset_id}: expected "
            f"TRAIN/VALIDATION/TEST coverage, "
            f"found {sorted(observed_splits)}."
        )

print(
    "✓ TRAIN / VALIDATION / TEST coverage "
    "verified for every dataset"
)


# ==================================================================================================
# 10. VALIDATE MANIFEST STATUS
# ==================================================================================================

status_values = (
    native_manifest[
        "status"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)

non_pass_status = native_manifest[
    status_values != "PASS"
]

if not non_pass_status.empty:

    raise RuntimeError(
        "Notebook 02 native manifest contains "
        "non-PASS records:\n"
        + non_pass_status[
            [
                "dataset_id",
                "split",
                "status",
            ]
        ].to_string(index=False)
    )

print(
    "✓ All Notebook 02 native manifest "
    "records have status=PASS"
)


# ==================================================================================================
# 11. VALIDATE PERSISTED NOTEBOOK 02 FLAGS
# ==================================================================================================

BOOLEAN_FLAG_COLUMNS = [
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "reload_validation",
]

for column in BOOLEAN_FLAG_COLUMNS:

    for index, value in native_manifest[
        column
    ].items():

        parsed = parse_boolean(
            value
        )

        if not parsed:

            dataset_id = native_manifest.loc[
                index,
                "dataset_id"
            ]

            split = native_manifest.loc[
                index,
                "split"
            ]

            raise RuntimeError(
                f"Notebook 02 integrity flag failed: "
                f"{column}={value!r} | "
                f"dataset={dataset_id} | "
                f"split={split}"
            )

print(
    "✓ Notebook 02 persisted integrity flags verified"
)


# ==================================================================================================
# 12. SELECT TRAIN RECORDS
# ==================================================================================================

train_manifest = native_manifest[
    native_manifest[
        "split"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    == "train"
].copy()

if len(train_manifest) != (
    len(DATASET_IDS)
):

    raise RuntimeError(
        f"Expected {len(DATASET_IDS)} TRAIN "
        f"manifest records, "
        f"found {len(train_manifest)}."
    )

print()
print(
    f"✓ TRAIN records selected : "
    f"{len(train_manifest)}"
)


# ==================================================================================================
# 13. VALIDATE TRAIN MANIFEST UNIQUENESS
# ==================================================================================================

train_duplicates = train_manifest[
    train_manifest[
        "dataset_id"
    ].duplicated(keep=False)
]

if not train_duplicates.empty:

    raise RuntimeError(
        "Duplicate TRAIN manifest records detected:\n"
        + train_duplicates[
            [
                "dataset_id",
                "split",
                "relative_path",
            ]
        ].to_string(index=False)
    )

print(
    "✓ No duplicate TRAIN manifest records"
)


# ==================================================================================================
# 14. INITIALIZE RUNTIME CONTAINERS
# ==================================================================================================

TRAINING_DATA = {}

TRAINING_METADATA = {}

TRAINING_FEATURE_COLUMNS = {}

TRAINING_GENERATIVE_COLUMNS = {}

TRAINING_TARGET_COLUMNS = {}

TRAINING_PROVENANCE_COLUMNS = {}

TRAINING_IDENTIFIER_COLUMNS = {}

print()
print(
    "Runtime containers initialized:"
)

print(
    "  ✓ TRAINING_DATA"
)

print(
    "  ✓ TRAINING_METADATA"
)

print(
    "  ✓ TRAINING_FEATURE_COLUMNS"
)

print(
    "  ✓ TRAINING_GENERATIVE_COLUMNS"
)

print(
    "  ✓ TRAINING_TARGET_COLUMNS"
)

print(
    "  ✓ TRAINING_PROVENANCE_COLUMNS"
)

print(
    "  ✓ TRAINING_IDENTIFIER_COLUMNS"
)


# ==================================================================================================
# 15. LOAD AND VALIDATE CANONICAL TRAINING DATA
# ==================================================================================================

print()
print("=" * 100)
print("LOADING CANONICAL NOTEBOOK 02 TRAIN DATA")
print("=" * 100)

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(
        f"DATASET : {dataset_id}"
    )
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Locate TRAIN record
    # ----------------------------------------------------------------------------------------------

    train_record = train_manifest[
        train_manifest[
            "dataset_id"
        ].astype(str)
        == dataset_id
    ]

    if len(train_record) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one "
            f"TRAIN manifest record, "
            f"found {len(train_record)}."
        )

    record = train_record.iloc[0]

    # ----------------------------------------------------------------------------------------------
    # Manifest metadata
    # ----------------------------------------------------------------------------------------------

    split = (
        str(record["split"])
        .strip()
        .lower()
    )

    if split != "train":

        raise RuntimeError(
            f"{dataset_id}: selected manifest "
            f"record is not TRAIN."
        )

    target_column = (
        str(record["target_column"])
        .strip()
    )

    provenance_column = (
        str(record["provenance_column"])
        .strip()
    )

    manifest_identifiers = (
        parse_identifier_columns(
            record[
                "identifier_columns"
            ]
        )
    )

    manifest_feature_count = (
        parse_manifest_count(
            record[
                "preprocessing_feature_columns"
            ],
            f"{dataset_id}: "
            f"preprocessing_feature_columns",
        )
    )

    manifest_generative_count = (
        parse_manifest_count(
            record[
                "generative_columns"
            ],
            f"{dataset_id}: "
            f"generative_columns",
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Expected registry
    # ----------------------------------------------------------------------------------------------

    expected_target = (
        EXPECTED_DATASETS[
            dataset_id
        ]["target"]
    )

    expected_identifiers = (
        EXPECTED_DATASETS[
            dataset_id
        ]["identifiers"]
    )

    if target_column != expected_target:

        raise RuntimeError(
            f"{dataset_id}: target mismatch.\n"
            f"Expected: {expected_target}\n"
            f"Found   : {target_column}"
        )

    if manifest_identifiers != (
        expected_identifiers
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier registry mismatch.\n"
            f"Expected: {expected_identifiers}\n"
            f"Found   : {manifest_identifiers}"
        )

    # ----------------------------------------------------------------------------------------------
    # Resolve canonical TRAIN file
    # ----------------------------------------------------------------------------------------------

    train_path = resolve_manifest_path(
        record["relative_path"],
        record["absolute_path"],
    )

    print(
        f"  ✓ Split               : "
        f"{split}"
    )

    print(
        f"  ✓ Target column       : "
        f"{target_column}"
    )

    print(
        f"  ✓ Native TRAIN file  : "
        f"{train_path}"
    )

    # ----------------------------------------------------------------------------------------------
    # File-size integrity
    # ----------------------------------------------------------------------------------------------

    actual_file_size = (
        train_path.stat().st_size
    )

    expected_file_size = (
        parse_manifest_count(
            record["file_size_bytes"],
            f"{dataset_id}: file_size_bytes",
        )
    )

    if actual_file_size != (
        expected_file_size
    ):

        raise RuntimeError(
            f"{dataset_id}: file size mismatch.\n"
            f"Expected: {expected_file_size}\n"
            f"Found   : {actual_file_size}"
        )

    print(
        f"  ✓ File size           : "
        f"{actual_file_size:,} bytes"
    )

    # ----------------------------------------------------------------------------------------------
    # SHA-256 integrity
    # ----------------------------------------------------------------------------------------------

    expected_sha256 = (
        str(
            record["sha256"]
        )
        .strip()
        .lower()
    )

    actual_sha256 = (
        sha256_file(
            train_path
        )
        .lower()
    )

    if actual_sha256 != (
        expected_sha256
    ):

        raise RuntimeError(
            f"{dataset_id}: SHA-256 mismatch.\n"
            f"Expected: {expected_sha256}\n"
            f"Found   : {actual_sha256}"
        )

    print(
        f"  ✓ SHA-256             : "
        f"{actual_sha256}"
    )

    # ----------------------------------------------------------------------------------------------
    # Load TRAIN data
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(
        train_path
    )

    expected_rows = (
        parse_manifest_count(
            record["rows"],
            f"{dataset_id}: rows",
        )
    )

    expected_native_columns = (
        parse_manifest_count(
            record["columns"],
            f"{dataset_id}: columns",
        )
    )

    if len(df) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: row count mismatch.\n"
            f"Expected: {expected_rows}\n"
            f"Found   : {len(df)}"
        )

    if len(df.columns) != (
        expected_native_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: column count mismatch.\n"
            f"Expected: {expected_native_columns}\n"
            f"Found   : {len(df.columns)}"
        )

    print(
        f"  ✓ Loaded TRAIN data  : "
        f"{len(df):,} rows × "
        f"{len(df.columns):,} columns"
    )

    print(
        f"  ✓ Row count           : "
        f"{len(df):,}"
    )

    print(
        f"  ✓ Column count        : "
        f"{len(df.columns):,}"
    )

    # ----------------------------------------------------------------------------------------------
    # Column uniqueness
    # ----------------------------------------------------------------------------------------------

    if not df.columns.is_unique:

        duplicated_columns = (
            df.columns[
                df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate column "
            f"names found:\n"
            f"{duplicated_columns}"
        )

    print(
        "  ✓ Column names unique"
    )

    # ----------------------------------------------------------------------------------------------
    # Target presence
    # ----------------------------------------------------------------------------------------------

    if target_column not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column missing."
        )

    print(
        f"  ✓ Target present      : "
        f"{target_column}"
    )

    # ----------------------------------------------------------------------------------------------
    # Provenance presence
    # ----------------------------------------------------------------------------------------------

    if provenance_column not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column missing."
        )

    print(
        f"  ✓ Provenance present   : "
        f"{provenance_column}"
    )

    # ----------------------------------------------------------------------------------------------
    # Explicit identifier exclusion
    # ----------------------------------------------------------------------------------------------

    present_identifiers = [
        column
        for column in expected_identifiers
        if column in df.columns
    ]

    if present_identifiers:

        raise RuntimeError(
            f"{dataset_id}: explicit identifiers "
            f"are present in native TRAIN data:\n"
            f"{present_identifiers}"
        )

    print(
        "  ✓ Identifier exclusion verified: "
        + (
            "none"
            if not expected_identifiers
            else str(expected_identifiers)
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Derive generative columns from ACTUAL native schema
    # ----------------------------------------------------------------------------------------------

    actual_columns = list(
        df.columns
    )

    actual_generative_columns = [
        column
        for column in actual_columns
        if column != provenance_column
        and column not in expected_identifiers
    ]

    actual_generative_count = len(
        actual_generative_columns
    )

    if actual_generative_count != (
        manifest_generative_count
    ):

        raise RuntimeError(
            f"{dataset_id}: generative column count mismatch.\n"
            f"Manifest count: {manifest_generative_count}\n"
            f"Native count  : {actual_generative_count}"
        )

    generative_columns = (
        actual_generative_columns.copy()
    )

    print(
        f"  ✓ Generative columns : "
        f"{len(generative_columns)}"
    )

    # ----------------------------------------------------------------------------------------------
    # Target retained in generative schema
    # ----------------------------------------------------------------------------------------------

    if target_column not in (
        generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: target is missing "
            f"from generative schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Derive feature columns
    # ----------------------------------------------------------------------------------------------

    feature_columns = [
        column
        for column in generative_columns
        if column != target_column
    ]

    if len(feature_columns) != (
        manifest_feature_count
    ):

        raise RuntimeError(
            f"{dataset_id}: feature column count mismatch.\n"
            f"Manifest count: {manifest_feature_count}\n"
            f"Native count  : {len(feature_columns)}"
        )

    print(
        f"  ✓ Feature columns    : "
        f"{len(feature_columns)}"
    )

    # ----------------------------------------------------------------------------------------------
    # Schema count identity
    # ----------------------------------------------------------------------------------------------

    if (
        len(feature_columns)
        + 1
        != len(generative_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: feature + target "
            f"does not equal generative schema."
        )

    if (
        len(generative_columns)
        + 1
        != len(df.columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: generative + provenance "
            f"does not equal native schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Generative schema membership
    # ----------------------------------------------------------------------------------------------

    if not set(
        generative_columns
    ).issubset(
        set(actual_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: generative schema contains "
            f"columns missing from native TRAIN data."
        )

    print(
        "  ✓ Generative schema membership verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Native schema order
    # ----------------------------------------------------------------------------------------------

    provenance_positions = [
        index
        for index, column in enumerate(
            actual_columns
        )
        if column == provenance_column
    ]

    if len(provenance_positions) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one "
            f"provenance position, "
            f"found {provenance_positions}"
        )

    provenance_position = (
        provenance_positions[0]
    )

    print(
        f"  ✓ Native schema order verified "
        f"(provenance position={provenance_position})"
    )

    # ----------------------------------------------------------------------------------------------
    # Target uniqueness
    # ----------------------------------------------------------------------------------------------

    target_occurrences = sum(
        column == target_column
        for column in actual_columns
    )

    if target_occurrences != 1:

        raise RuntimeError(
            f"{dataset_id}: target column occurs "
            f"{target_occurrences} times."
        )

    print(
        "  ✓ Target uniqueness verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Provenance uniqueness
    # ----------------------------------------------------------------------------------------------

    provenance_occurrences = sum(
        column == provenance_column
        for column in actual_columns
    )

    if provenance_occurrences != 1:

        raise RuntimeError(
            f"{dataset_id}: provenance column occurs "
            f"{provenance_occurrences} times."
        )

    print(
        "  ✓ Provenance uniqueness verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Provenance integrity
    # ----------------------------------------------------------------------------------------------

    provenance_series = df[
        provenance_column
    ]

    if provenance_series.isna().any():

        raise RuntimeError(
            f"{dataset_id}: provenance contains "
            f"missing values."
        )

    if not provenance_series.is_unique:

        raise RuntimeError(
            f"{dataset_id}: provenance values "
            f"are not unique."
        )

    print(
        "  ✓ Provenance integrity verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Target non-empty
    # ----------------------------------------------------------------------------------------------

    if df[
        target_column
    ].dropna().empty:

        raise RuntimeError(
            f"{dataset_id}: target data is empty."
        )

    print(
        "  ✓ Target data is non-empty"
    )

    # ----------------------------------------------------------------------------------------------
    # Notebook 02 schema flags
    # ----------------------------------------------------------------------------------------------

    for flag_column in (
        BOOLEAN_FLAG_COLUMNS
    ):

        if not parse_boolean(
            record[flag_column]
        ):

            raise RuntimeError(
                f"{dataset_id}: Notebook 02 "
                f"flag {flag_column} is not TRUE."
            )

    print(
        "  ✓ Notebook 02 schema flags verified"
    )

    # ==============================================================================================
    # CRITICAL RUNTIME CONTAINER PERSISTENCE
    # ==============================================================================================

    TRAINING_DATA[
        dataset_id
    ] = df

    TRAINING_FEATURE_COLUMNS[
        dataset_id
    ] = feature_columns.copy()

    TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ] = generative_columns.copy()

    TRAINING_TARGET_COLUMNS[
        dataset_id
    ] = target_column

    TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ] = provenance_column

    TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ] = expected_identifiers.copy()

    TRAINING_METADATA[
        dataset_id
    ] = {
        "dataset_id": dataset_id,
        "split": split,
        "native_path": str(train_path),
        "rows": len(df),
        "native_columns": len(df.columns),
        "feature_columns": feature_columns.copy(),
        "generative_columns": generative_columns.copy(),
        "target_column": target_column,
        "provenance_column": provenance_column,
        "identifier_columns": expected_identifiers.copy(),
        "provenance_position": provenance_position,
        "file_size_bytes": actual_file_size,
        "sha256": actual_sha256,
        "manifest_feature_count": manifest_feature_count,
        "manifest_generative_count": manifest_generative_count,
    }

    print(
        f"  ✓ {dataset_id} successfully validated"
    )


# ==================================================================================================
# 16. FINAL TRAINING DATA COVERAGE
# ==================================================================================================

print()
print("=" * 100)
print("FINAL TRAINING DATA COVERAGE")
print("=" * 100)

if set(
    TRAINING_DATA.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_DATA dataset coverage mismatch.\n"
        f"Expected: {DATASET_IDS}\n"
        f"Found   : {list(TRAINING_DATA.keys())}"
    )

print(
    "✓ Loaded datasets: "
    + ", ".join(DATASET_IDS)
)


# ==================================================================================================
# 17. FINAL RUNTIME SCHEMA CONTAINER COMPLETENESS
# ==================================================================================================

print()
print("-" * 100)
print("FINAL RUNTIME SCHEMA CONTAINER COMPLETENESS")
print("-" * 100)

EXPECTED_SCHEMA_CONTAINERS = {
    "TRAINING_DATA": TRAINING_DATA,
    "TRAINING_METADATA": TRAINING_METADATA,
    "TRAINING_FEATURE_COLUMNS": TRAINING_FEATURE_COLUMNS,
    "TRAINING_GENERATIVE_COLUMNS": TRAINING_GENERATIVE_COLUMNS,
    "TRAINING_TARGET_COLUMNS": TRAINING_TARGET_COLUMNS,
    "TRAINING_PROVENANCE_COLUMNS": TRAINING_PROVENANCE_COLUMNS,
    "TRAINING_IDENTIFIER_COLUMNS": TRAINING_IDENTIFIER_COLUMNS,
}

for container_name, container in (
    EXPECTED_SCHEMA_CONTAINERS.items()
):

    missing_datasets = sorted(
        set(DATASET_IDS)
        - set(container.keys())
    )

    unexpected_datasets = sorted(
        set(container.keys())
        - set(DATASET_IDS)
    )

    if missing_datasets:

        raise RuntimeError(
            f"{container_name} is missing dataset entries: "
            f"{missing_datasets}"
        )

    if unexpected_datasets:

        raise RuntimeError(
            f"{container_name} contains unexpected dataset entries: "
            f"{unexpected_datasets}"
        )

    if len(container) != len(DATASET_IDS):

        raise RuntimeError(
            f"{container_name}: expected "
            f"{len(DATASET_IDS)} entries, "
            f"found {len(container)}."
        )

    print(
        f"✓ {container_name:<35} "
        f"{len(container)}/{len(DATASET_IDS)} datasets present"
    )


# ==================================================================================================
# 18. FINAL METADATA VERIFICATION
# ==================================================================================================

print()
print("-" * 100)
print("FINAL METADATA VERIFICATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[
        dataset_id
    ]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    identifiers = TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ]

    metadata = TRAINING_METADATA[
        dataset_id
    ]

    if metadata["rows"] != len(df):

        raise RuntimeError(
            f"{dataset_id}: metadata row count mismatch."
        )

    if metadata["native_columns"] != len(
        df.columns
    ):

        raise RuntimeError(
            f"{dataset_id}: metadata native column count mismatch."
        )

    if metadata["feature_columns"] != features:

        raise RuntimeError(
            f"{dataset_id}: metadata feature schema mismatch."
        )

    if metadata["generative_columns"] != (
        generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: metadata generative schema mismatch."
        )

    if metadata["target_column"] != target:

        raise RuntimeError(
            f"{dataset_id}: metadata target mismatch."
        )

    if metadata["provenance_column"] != provenance:

        raise RuntimeError(
            f"{dataset_id}: metadata provenance mismatch."
        )

    if metadata["identifier_columns"] != identifiers:

        raise RuntimeError(
            f"{dataset_id}: metadata identifier mismatch."
        )

    print(
        f"✓ {dataset_id:<20} "
        f"rows={len(df):,} | "
        f"native_cols={len(df.columns):>2} | "
        f"features={len(features):>2} | "
        f"generative={len(generative_columns):>2} | "
        f"target={target}"
    )


# ==================================================================================================
# 19. RUNTIME OBJECT VERIFICATION
# ==================================================================================================

print()
print("-" * 100)
print("RUNTIME OBJECT VERIFICATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    if not isinstance(
        TRAINING_DATA[dataset_id],
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_DATA is not "
            f"a pandas DataFrame."
        )

    if not isinstance(
        TRAINING_FEATURE_COLUMNS[dataset_id],
        list,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_FEATURE_COLUMNS "
            f"is not a list."
        )

    if not isinstance(
        TRAINING_GENERATIVE_COLUMNS[dataset_id],
        list,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_GENERATIVE_COLUMNS "
            f"is not a list."
        )

    if not isinstance(
        TRAINING_TARGET_COLUMNS[dataset_id],
        str,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_TARGET_COLUMNS "
            f"is not a string."
        )

    if not isinstance(
        TRAINING_PROVENANCE_COLUMNS[dataset_id],
        str,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_PROVENANCE_COLUMNS "
            f"is not a string."
        )

    if not isinstance(
        TRAINING_IDENTIFIER_COLUMNS[dataset_id],
        list,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_IDENTIFIER_COLUMNS "
            f"is not a list."
        )

print(
    "✓ TRAINING_DATA verified"
)

print(
    "✓ TRAINING_METADATA verified"
)

print(
    "✓ All schema runtime containers verified"
)


# ==================================================================================================
# 20. FINAL RESEARCH-POLICY VERIFICATION
# ==================================================================================================

print()
print("-" * 100)
print("FINAL RESEARCH-POLICY VERIFICATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[
        dataset_id
    ]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    identifiers = TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ]

    if target in features:

        raise RuntimeError(
            f"{dataset_id}: target leakage detected."
        )

    if provenance in features:

        raise RuntimeError(
            f"{dataset_id}: provenance leakage detected."
        )

    if provenance in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance included "
            f"in generative schema."
        )

    if set(
        identifiers
    ).intersection(
        features
    ):

        raise RuntimeError(
            f"{dataset_id}: identifiers included "
            f"in feature schema."
        )

    if set(
        identifiers
    ).intersection(
        generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: identifiers included "
            f"in generative schema."
        )

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target missing "
            f"from generative schema."
        )

    if len(features) != (
        len(generative_columns) - 1
    ):

        raise RuntimeError(
            f"{dataset_id}: feature/generative "
            f"schema relationship invalid."
        )

print(
    "✓ Target excluded from feature predictor schema"
)

print(
    "✓ Target retained in generative schema"
)

print(
    "✓ Provenance retained for auditability"
)

print(
    "✓ Explicit identifiers excluded"
)

print(
    "✓ No target/provenance/identifier leakage"
)

print(
    "✓ TRAIN-only policy verified"
)


# ==================================================================================================
# 21. FINAL TRAINING DATA SUMMARY
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 03 — COMPLETION SUMMARY")
print("=" * 100)

total_train_rows = sum(
    len(
        TRAINING_DATA[dataset_id]
    )
    for dataset_id in DATASET_IDS
)

print(
    f"Project root                 : "
    f"{PROJECT_ROOT}"
)

print(
    f"Notebook 02 root             : "
    f"{NB02_ROOT}"
)

print(
    f"Native manifest              : "
    f"{NATIVE_MANIFEST_PATH}"
)

print(
    f"TRAIN datasets loaded        : "
    f"{len(TRAINING_DATA)}"
)

print(
    f"Total TRAIN rows loaded      : "
    f"{total_train_rows:,}"
)

print()
print("Dataset summary:")

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[
        dataset_id
    ]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    provenance_position = (
        list(df.columns).index(
            provenance
        )
    )

    print(
        f"  {dataset_id:<20} "
        f"rows={len(df):,} | "
        f"native={len(df.columns):>2} | "
        f"features={len(features):>2} | "
        f"generative={len(generative_columns):>2} | "
        f"target={target} | "
        f"provenance_pos={provenance_position}"
    )


# ==================================================================================================
# 22. FINAL RESEARCH-POLICY SUMMARY
# ==================================================================================================

print()
print("Research-policy verification:")

print(
    "  ✓ Canonical Notebook 02 native TRAIN data used"
)

print(
    "  ✓ Raw datasets not reloaded"
)

print(
    "  ✓ Notebook 02 preprocessing not refitted"
)

print(
    "  ✓ Encoded/scaled matrices not used"
)

print(
    "  ✓ Only TRAIN split selected"
)

print(
    "  ✓ Validation split excluded"
)

print(
    "  ✓ Test split excluded"
)

print(
    "  ✓ Target retained in generative schema"
)

print(
    "  ✓ Target excluded from feature predictor schema"
)

print(
    "  ✓ Provenance retained for auditability"
)

print(
    "  ✓ Explicit identifiers excluded"
)

print(
    "  ✓ File-size integrity verified"
)

print(
    "  ✓ SHA-256 integrity verified"
)

print(
    "  ✓ Native column order preserved"
)

print(
    "  ✓ TRAIN dataset coverage verified"
)

print(
    "  ✓ Runtime schema containers complete"
)

print(
    "  ✓ Runtime objects verified"
)


# ==================================================================================================
# 23. MEMORY CLEANUP
# ==================================================================================================

gc.collect()


# ==================================================================================================
# FINAL STATUS
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 03 STATUS: PASS")
print("=" * 100)

SECTION 03 — LOAD PROCESSED TRAINING DATA
✓ Google Drive     : /content/drive
✓ MyDrive          : /content/drive/MyDrive
✓ Project root     : /content/drive/MyDrive/SPP_GAN_Research

✓ Notebook 02 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native data root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ Native manifest  : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv

Expected datasets:
  • adult_income         target = income
  • bank_marketing       target = y
  • diabetes_130us       target = readmitted

----------------------------------------------------------------------------------------------------
LOAD NOTEBOOK 02 NATIVE MANIFEST
----------------------------------------------------------------------------------------------------
✓ Native manifest loaded
  Rows    : 9
  Columns : 19
✓ Exact required manifest schema verified
✓ Manifest row count verified: 9
✓ D

In [27]:
# ==================================================================================================
# 4. VALIDATE INPUT SCHEMA
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — VALIDATE INPUT SCHEMA")
print("=" * 100)

INPUT_SCHEMA_SUMMARY = {}

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[dataset_id]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    identifiers = TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ]

    # ----------------------------------------------------------------------------------------------
    # Basic existence
    # ----------------------------------------------------------------------------------------------

    if not set(features).issubset(df.columns):

        raise RuntimeError(
            f"{dataset_id}: preprocessing feature columns missing."
        )

    if not set(generative_columns).issubset(df.columns):

        raise RuntimeError(
            f"{dataset_id}: generative columns missing."
        )

    if target not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column missing."
        )

    if provenance not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column missing."
        )

    # ----------------------------------------------------------------------------------------------
    # Frozen Notebook 02 policy
    # ----------------------------------------------------------------------------------------------

    if target in features:

        raise RuntimeError(
            f"{dataset_id}: TARGET LEAKAGE — target appears in features."
        )

    if provenance in features:

        raise RuntimeError(
            f"{dataset_id}: PROVENANCE LEAKAGE."
        )

    identifier_overlap = (
        set(features)
        .intersection(identifiers)
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: IDENTIFIER LEAKAGE:\n"
            f"{sorted(identifier_overlap)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Generative schema
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target is not retained in generative schema."
        )

    if provenance in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance appears in generative schema."
        )

    generative_identifier_overlap = (
        set(generative_columns)
        .intersection(identifiers)
    )

    if generative_identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifiers appear in generative schema:\n"
            f"{sorted(generative_identifier_overlap)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Target registry consistency
    # ----------------------------------------------------------------------------------------------
    # TRAINING_TARGET_COLUMNS is the canonical target mapping loaded from
    # Notebook 02 artifacts in Section 03.
    # No undefined TARGET_COLUMNS reference is used here.

    if target != TRAINING_TARGET_COLUMNS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: target registry mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Store validated schema summary
    # ----------------------------------------------------------------------------------------------

    INPUT_SCHEMA_SUMMARY[dataset_id] = {
        "training_rows": len(df),
        "native_columns": len(df.columns),
        "feature_columns": len(features),
        "generative_columns": len(generative_columns),
        "target_column": target,
        "provenance_column": provenance,
        "identifier_columns": identifiers,
    }

    print(
        f"✓ {dataset_id:<20} | "
        f"features={len(features):>3} | "
        f"generative={len(generative_columns):>3} | "
        f"target={target} | "
        f"target/provenance/ID separation PASS"
    )

print()
print("✓ SECTION 4 — INPUT SCHEMA : PASS")

SECTION 4 — VALIDATE INPUT SCHEMA
✓ adult_income         | features= 14 | generative= 15 | target=income | target/provenance/ID separation PASS
✓ bank_marketing       | features= 16 | generative= 17 | target=y | target/provenance/ID separation PASS
✓ diabetes_130us       | features= 47 | generative= 48 | target=readmitted | target/provenance/ID separation PASS

✓ SECTION 4 — INPUT SCHEMA : PASS


In [29]:
# ==================================================================================================
# 5. DEFINE EXACT STATISTICAL BASELINE METHODS
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — DEFINE EXACT STATISTICAL BASELINE METHODS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Exact Baseline Execution Registry
# --------------------------------------------------------------------------------------------------

BASELINE_METHODS = [
    "independent_marginal",
    "gaussian_copula",
]

# --------------------------------------------------------------------------------------------------
# 2. Exact Baseline Definitions
# --------------------------------------------------------------------------------------------------

BASELINE_DEFINITIONS = {

    "independent_marginal": {

        "name": "Independent Marginal Sampling",

        "family": "Statistical",

        "principle": (
            "Estimate the empirical marginal distribution of each "
            "generative variable independently and sample each "
            "variable independently."
        ),

        "dependency_model": "None",

        "privacy_guarantee": False,
    },

    "gaussian_copula": {

        "name": "Gaussian Copula",

        "family": "Statistical",

        "principle": (
            "Model individual variable distributions together with "
            "their dependence structure through a Gaussian copula."
        ),

        "dependency_model": "Gaussian copula",

        "privacy_guarantee": False,
    },
}

# --------------------------------------------------------------------------------------------------
# 3. Registry Integrity Validation
# --------------------------------------------------------------------------------------------------

if not isinstance(BASELINE_METHODS, list):
    raise RuntimeError(
        "BASELINE_METHODS must be a list."
    )

if len(BASELINE_METHODS) == 0:
    raise RuntimeError(
        "BASELINE_METHODS is empty."
    )

if len(BASELINE_METHODS) != len(set(BASELINE_METHODS)):
    raise RuntimeError(
        "BASELINE_METHODS contains duplicate baseline identifiers."
    )

missing_definitions = [
    baseline_name
    for baseline_name in BASELINE_METHODS
    if baseline_name not in BASELINE_DEFINITIONS
]

if missing_definitions:
    raise RuntimeError(
        f"Missing baseline definitions: {missing_definitions}"
    )

extra_definitions = [
    baseline_name
    for baseline_name in BASELINE_DEFINITIONS
    if baseline_name not in BASELINE_METHODS
]

if extra_definitions:
    raise RuntimeError(
        f"Baseline definitions not present in execution registry: {extra_definitions}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Validate Required Definition Fields
# --------------------------------------------------------------------------------------------------

REQUIRED_BASELINE_FIELDS = {
    "name",
    "family",
    "principle",
    "dependency_model",
    "privacy_guarantee",
}

for baseline_name in BASELINE_METHODS:

    definition = BASELINE_DEFINITIONS[baseline_name]

    missing_fields = REQUIRED_BASELINE_FIELDS.difference(
        definition.keys()
    )

    if missing_fields:
        raise RuntimeError(
            f"{baseline_name}: missing definition fields: "
            f"{sorted(missing_fields)}"
        )

    if definition["family"] != "Statistical":
        raise RuntimeError(
            f"{baseline_name}: baseline family must be 'Statistical'."
        )

    if not isinstance(definition["privacy_guarantee"], bool):
        raise RuntimeError(
            f"{baseline_name}: privacy_guarantee must be boolean."
        )

# --------------------------------------------------------------------------------------------------
# 5. Display Frozen Baseline Registry
# --------------------------------------------------------------------------------------------------

print()
print("FROZEN STATISTICAL BASELINE REGISTRY")
print("-" * 100)

for baseline_index, baseline_name in enumerate(BASELINE_METHODS, start=1):

    definition = BASELINE_DEFINITIONS[baseline_name]

    print()
    print(
        f"{baseline_index}. {baseline_name}"
    )
    print(
        f"   Name         : {definition['name']}"
    )
    print(
        f"   Family       : {definition['family']}"
    )
    print(
        f"   Dependency   : {definition['dependency_model']}"
    )
    print(
        f"   DP guarantee : {definition['privacy_guarantee']}"
    )
    print(
        f"   Principle    : {definition['principle']}"
    )

# --------------------------------------------------------------------------------------------------
# 6. Final Section 5 Integrity Gate
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if BASELINE_METHODS != EXPECTED_BASELINES:
    raise RuntimeError(
        "Baseline registry does not match the frozen research configuration."
    )

for baseline_name in EXPECTED_BASELINES:

    if baseline_name not in BASELINE_DEFINITIONS:
        raise RuntimeError(
            f"Required baseline missing: {baseline_name}"
        )

print()
print("-" * 100)
print(f"✓ Statistical baselines defined : {len(BASELINE_METHODS)}")
print(f"✓ Baseline identifiers           : {BASELINE_METHODS}")
print("✓ Registry integrity             : PASS")
print("✓ Exact baseline registry frozen.")

SECTION 5 — DEFINE EXACT STATISTICAL BASELINE METHODS

FROZEN STATISTICAL BASELINE REGISTRY
----------------------------------------------------------------------------------------------------

1. independent_marginal
   Name         : Independent Marginal Sampling
   Family       : Statistical
   Dependency   : None
   DP guarantee : False
   Principle    : Estimate the empirical marginal distribution of each generative variable independently and sample each variable independently.

2. gaussian_copula
   Name         : Gaussian Copula
   Family       : Statistical
   Dependency   : Gaussian copula
   DP guarantee : False
   Principle    : Model individual variable distributions together with their dependence structure through a Gaussian copula.

----------------------------------------------------------------------------------------------------
✓ Statistical baselines defined : 2
✓ Baseline identifiers           : ['independent_marginal', 'gaussian_copula']
✓ Registry integrity       

In [31]:
# ==================================================================================================
# 6. CONFIGURE STATISTICAL BASELINES
# ==================================================================================================

print("=" * 100)
print("SECTION 6 — CONFIGURE STATISTICAL BASELINES")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Exact Baseline Configuration
# --------------------------------------------------------------------------------------------------

BASELINE_CONFIG = {

    "independent_marginal": {

        "sampling": "empirical_distribution",

        "replacement": True,

        "sample_size": "training_rows",

        "fit_data": "native_train_only",
    },

    "gaussian_copula": {

        "default_distribution": "norm",

        "enforce_min_max_values": True,

        "enforce_rounding": True,

        "sample_size": "training_rows",

        "fit_data": "native_train_only",
    },
}

# --------------------------------------------------------------------------------------------------
# 2. Configuration Registry Integrity
# --------------------------------------------------------------------------------------------------

if not isinstance(BASELINE_CONFIG, dict):
    raise RuntimeError(
        "BASELINE_CONFIG must be a dictionary."
    )

if len(BASELINE_CONFIG) == 0:
    raise RuntimeError(
        "BASELINE_CONFIG is empty."
    )

# Every baseline defined in Section 5 must have a configuration.
missing_configurations = [
    baseline_name
    for baseline_name in BASELINE_METHODS
    if baseline_name not in BASELINE_CONFIG
]

if missing_configurations:
    raise RuntimeError(
        "Missing configurations for baseline(s): "
        f"{missing_configurations}"
    )

# No configuration may exist for an undefined baseline.
unexpected_configurations = [
    baseline_name
    for baseline_name in BASELINE_CONFIG
    if baseline_name not in BASELINE_METHODS
]

if unexpected_configurations:
    raise RuntimeError(
        "Unexpected baseline configuration(s): "
        f"{unexpected_configurations}"
    )

# --------------------------------------------------------------------------------------------------
# 3. Required Configuration Fields
# --------------------------------------------------------------------------------------------------

REQUIRED_CONFIG_FIELDS = {

    "independent_marginal": {
        "sampling",
        "replacement",
        "sample_size",
        "fit_data",
    },

    "gaussian_copula": {
        "default_distribution",
        "enforce_min_max_values",
        "enforce_rounding",
        "sample_size",
        "fit_data",
    },
}

for baseline_name in BASELINE_METHODS:

    configuration = BASELINE_CONFIG[baseline_name]

    required_fields = REQUIRED_CONFIG_FIELDS[baseline_name]

    missing_fields = required_fields.difference(
        configuration.keys()
    )

    if missing_fields:
        raise RuntimeError(
            f"{baseline_name}: missing configuration field(s): "
            f"{sorted(missing_fields)}"
        )

# --------------------------------------------------------------------------------------------------
# 4. Independent Marginal Configuration Validation
# --------------------------------------------------------------------------------------------------

independent_config = BASELINE_CONFIG["independent_marginal"]

if independent_config["sampling"] != "empirical_distribution":

    raise RuntimeError(
        "Independent Marginal Sampling must use "
        "'empirical_distribution'."
    )

if independent_config["replacement"] is not True:

    raise RuntimeError(
        "Independent Marginal Sampling must use sampling with replacement."
    )

if independent_config["sample_size"] != "training_rows":

    raise RuntimeError(
        "Independent Marginal Sampling must generate "
        "the training-row sample size."
    )

if independent_config["fit_data"] != "native_train_only":

    raise RuntimeError(
        "Independent Marginal Sampling must be fitted using "
        "native training data only."
    )

# --------------------------------------------------------------------------------------------------
# 5. Gaussian Copula Configuration Validation
# --------------------------------------------------------------------------------------------------

gaussian_config = BASELINE_CONFIG["gaussian_copula"]

if gaussian_config["default_distribution"] != "norm":

    raise RuntimeError(
        "Gaussian Copula default distribution must be 'norm'."
    )

if gaussian_config["enforce_min_max_values"] is not True:

    raise RuntimeError(
        "Gaussian Copula must enforce training-derived minimum "
        "and maximum value constraints."
    )

if gaussian_config["enforce_rounding"] is not True:

    raise RuntimeError(
        "Gaussian Copula must enforce rounding for appropriate "
        "integer-valued variables."
    )

if gaussian_config["sample_size"] != "training_rows":

    raise RuntimeError(
        "Gaussian Copula must generate the training-row sample size."
    )

if gaussian_config["fit_data"] != "native_train_only":

    raise RuntimeError(
        "Gaussian Copula must be fitted using native training data only."
    )

# --------------------------------------------------------------------------------------------------
# 6. Cross-Baseline Research Policy Validation
# --------------------------------------------------------------------------------------------------

for baseline_name in BASELINE_METHODS:

    configuration = BASELINE_CONFIG[baseline_name]

    # All statistical baselines must use training data only.
    if configuration["fit_data"] != "native_train_only":

        raise RuntimeError(
            f"{baseline_name}: research leakage policy violated. "
            "Baseline fitting must use native training data only."
        )

    # All baselines must generate training-sized synthetic datasets.
    if configuration["sample_size"] != "training_rows":

        raise RuntimeError(
            f"{baseline_name}: sample-size policy violated. "
            "Synthetic sample size must equal training rows."
        )

# --------------------------------------------------------------------------------------------------
# 7. Display Frozen Configuration
# --------------------------------------------------------------------------------------------------

print()
print("FROZEN STATISTICAL BASELINE CONFIGURATION")
print("-" * 100)

print(
    json.dumps(
        BASELINE_CONFIG,
        indent=2,
    )
)

# --------------------------------------------------------------------------------------------------
# 8. Final Configuration Integrity Gate
# --------------------------------------------------------------------------------------------------

if set(BASELINE_CONFIG.keys()) != set(BASELINE_METHODS):

    raise RuntimeError(
        "BASELINE_CONFIG keys do not exactly match BASELINE_METHODS."
    )

if len(BASELINE_CONFIG) != 2:

    raise RuntimeError(
        "Expected exactly 2 statistical baseline configurations."
    )

print()
print("-" * 100)
print(f"✓ Statistical baselines configured : {len(BASELINE_CONFIG)}")
print(f"✓ Configured identifiers            : {list(BASELINE_CONFIG.keys())}")
print("✓ Configuration completeness        : PASS")
print("✓ Method-specific validation        : PASS")
print("✓ Training-only policy              : PASS")
print("✓ Training-sized sampling policy    : PASS")
print("✓ Configuration integrity           : PASS")
print("✓ Baseline configuration frozen.")

SECTION 6 — CONFIGURE STATISTICAL BASELINES

FROZEN STATISTICAL BASELINE CONFIGURATION
----------------------------------------------------------------------------------------------------
{
  "independent_marginal": {
    "sampling": "empirical_distribution",
    "replacement": true,
    "sample_size": "training_rows",
    "fit_data": "native_train_only"
  },
  "gaussian_copula": {
    "default_distribution": "norm",
    "enforce_min_max_values": true,
    "enforce_rounding": true,
    "sample_size": "training_rows",
    "fit_data": "native_train_only"
  }
}

----------------------------------------------------------------------------------------------------
✓ Statistical baselines configured : 2
✓ Configured identifiers            : ['independent_marginal', 'gaussian_copula']
✓ Configuration completeness        : PASS
✓ Method-specific validation        : PASS
✓ Training-only policy              : PASS
✓ Training-sized sampling policy    : PASS
✓ Configuration integrity           : PA

In [33]:
# ==================================================================================================
# 7. SET REPRODUCIBLE SEEDS
# ==================================================================================================

print("=" * 100)
print("SECTION 7 — SET REPRODUCIBLE SEEDS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Required Imports
# --------------------------------------------------------------------------------------------------

import os
import random
import numpy as np

# --------------------------------------------------------------------------------------------------
# 2. Validate Notebook 00 Master Seed
# --------------------------------------------------------------------------------------------------

if "MASTER_SEED" not in globals():
    raise RuntimeError(
        "MASTER_SEED is not available. "
        "Load the frozen Notebook 00 configuration before Section 7."
    )

if not isinstance(MASTER_SEED, (int, np.integer)):
    raise RuntimeError(
        "MASTER_SEED must be an integer."
    )

MASTER_SEED = int(MASTER_SEED)

if MASTER_SEED != 2025:
    raise RuntimeError(
        f"Unexpected MASTER_SEED={MASTER_SEED}. "
        "Frozen Notebook 00 master seed is 2025."
    )

# --------------------------------------------------------------------------------------------------
# 3. Global Reproducibility Function
# --------------------------------------------------------------------------------------------------

def seed_everything(seed):

    seed = int(seed)

    random.seed(seed)

    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


# --------------------------------------------------------------------------------------------------
# 4. Deterministic Dataset/Baseline Seed Policy
# --------------------------------------------------------------------------------------------------

def get_baseline_seed(
    dataset_index,
    baseline_index,
):

    dataset_index = int(dataset_index)
    baseline_index = int(baseline_index)

    return (
        MASTER_SEED
        + ((dataset_index + 1) * 1000)
        + ((baseline_index + 1) * 100)
    )


# --------------------------------------------------------------------------------------------------
# 5. Validate Seed Policy
# --------------------------------------------------------------------------------------------------

if len(DATASET_IDS) == 0:
    raise RuntimeError(
        "DATASET_IDS is empty. Cannot establish dataset-level seed policy."
    )

if len(BASELINE_METHODS) == 0:
    raise RuntimeError(
        "BASELINE_METHODS is empty. Cannot establish baseline-level seed policy."
    )

baseline_seed_registry = {}

for dataset_index, dataset_id in enumerate(DATASET_IDS):

    baseline_seed_registry[dataset_id] = {}

    for baseline_index, baseline_name in enumerate(BASELINE_METHODS):

        seed = get_baseline_seed(
            dataset_index=dataset_index,
            baseline_index=baseline_index,
        )

        baseline_seed_registry[dataset_id][baseline_name] = seed

# --------------------------------------------------------------------------------------------------
# 6. Verify Seed Uniqueness
# --------------------------------------------------------------------------------------------------

all_baseline_seeds = [
    seed
    for dataset_seeds in baseline_seed_registry.values()
    for seed in dataset_seeds.values()
]

if len(all_baseline_seeds) != len(set(all_baseline_seeds)):
    raise RuntimeError(
        "Dataset/baseline seed collision detected."
    )

# --------------------------------------------------------------------------------------------------
# 7. Initialize Global Reproducibility
# --------------------------------------------------------------------------------------------------

seed_everything(
    MASTER_SEED
)

# --------------------------------------------------------------------------------------------------
# 8. Display Reproducibility Configuration
# --------------------------------------------------------------------------------------------------

print()
print("REPRODUCIBILITY CONFIGURATION")
print("-" * 100)

print(
    f"✓ MASTER_SEED : {MASTER_SEED}"
)

print(
    "✓ Python random seeded"
)

print(
    "✓ NumPy random seeded"
)

print(
    "✓ PYTHONHASHSEED configured"
)

print()
print("DATASET / BASELINE SEED REGISTRY")
print("-" * 100)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id}"
    )

    for baseline_name in BASELINE_METHODS:

        print(
            f"  {baseline_name:<25} : "
            f"{baseline_seed_registry[dataset_id][baseline_name]}"
        )

# --------------------------------------------------------------------------------------------------
# 9. Final Integrity Gate
# --------------------------------------------------------------------------------------------------

expected_seed_count = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

if len(all_baseline_seeds) != expected_seed_count:
    raise RuntimeError(
        "Unexpected number of dataset/baseline seeds."
    )

if len(set(all_baseline_seeds)) != expected_seed_count:
    raise RuntimeError(
        "Dataset/baseline seeds are not unique."
    )

print()
print("-" * 100)
print(f"✓ Master seed validated            : {MASTER_SEED}")
print(f"✓ Datasets covered                : {len(DATASET_IDS)}")
print(f"✓ Baselines covered               : {len(BASELINE_METHODS)}")
print(f"✓ Unique baseline seeds           : {len(all_baseline_seeds)}")
print("✓ Python/NumPy reproducibility    : PASS")
print("✓ Dataset/baseline seed policy    : PASS")
print("✓ Seed collision check             : PASS")
print("✓ Reproducibility configuration frozen.")

SECTION 7 — SET REPRODUCIBLE SEEDS

REPRODUCIBILITY CONFIGURATION
----------------------------------------------------------------------------------------------------
✓ MASTER_SEED : 2025
✓ Python random seeded
✓ NumPy random seeded
✓ PYTHONHASHSEED configured

DATASET / BASELINE SEED REGISTRY
----------------------------------------------------------------------------------------------------
adult_income
  independent_marginal      : 3125
  gaussian_copula           : 3225
bank_marketing
  independent_marginal      : 4125
  gaussian_copula           : 4225
diabetes_130us
  independent_marginal      : 5125
  gaussian_copula           : 5225

----------------------------------------------------------------------------------------------------
✓ Master seed validated            : 2025
✓ Datasets covered                : 3
✓ Baselines covered               : 2
✓ Unique baseline seeds           : 6
✓ Python/NumPy reproducibility    : PASS
✓ Dataset/baseline seed policy    : PASS
✓ Seed coll

In [36]:
# ==================================================================================================
# NOTEBOOK 04 — ENVIRONMENT SETUP FOR STATISTICAL BASELINES
# ==================================================================================================

import sys
import subprocess

print("=" * 100)
print("NOTEBOOK 04 — SDV ENVIRONMENT CHECK")
print("=" * 100)

print(f"Python version : {sys.version}")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sdv",
    ]
)

print()
print("✓ SDV installation completed.")
print("⚠ Restart the Colab runtime if pip reports dependency conflicts.")

NOTEBOOK 04 — SDV ENVIRONMENT CHECK
Python version : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

✓ SDV installation completed.
⚠ Restart the Colab runtime if pip reports dependency conflicts.


In [38]:
# ==================================================================================================
# 8. FIT BASELINE DISTRIBUTIONS
# ==================================================================================================

print("=" * 100)
print("SECTION 8 — FIT BASELINE DISTRIBUTIONS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Required Imports
# --------------------------------------------------------------------------------------------------

import gc
import time
import random
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------------------
# 2. Validate Required Runtime Dependencies
# --------------------------------------------------------------------------------------------------

try:

    import sdv
    from sdv.metadata import Metadata
    from sdv.single_table import GaussianCopulaSynthesizer

except ImportError as exc:

    raise RuntimeError(
        "SDV is required for the Gaussian Copula baseline but is not installed "
        "in the current runtime. Install the validated SDV environment before "
        "running Notebook 04 Section 8."
    ) from exc

print(
    f"✓ SDV version    : "
    f"{getattr(sdv, '__version__', 'unknown')}"
)

print(
    f"✓ Python version : "
    f"{sys.version.split()[0]}"
)

# --------------------------------------------------------------------------------------------------
# 3. Define Canonical Baseline Artifact Root
# --------------------------------------------------------------------------------------------------

NB04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

NB04_BASELINE_ROOT = (
    NB04_ROOT
    / "baselines"
)

NB04_METADATA_ROOT = (
    NB04_BASELINE_ROOT
    / "metadata"
)

NB04_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    f"✓ Baseline artifact root : {NB04_BASELINE_ROOT}"
)

print(
    f"✓ Metadata artifact root : {NB04_METADATA_ROOT}"
)

# --------------------------------------------------------------------------------------------------
# 4. Independent Marginal Fitter
# --------------------------------------------------------------------------------------------------

def fit_independent_marginal(
    dataframe,
    columns,
):

    fitted = {}

    for column in columns:

        series = dataframe[column]

        # Preserve missing values as an explicit empirical state.
        values = (
            series.astype(object)
            .where(
                series.notna(),
                "__NB04_MISSING__",
            )
        )

        frequencies = (
            values
            .value_counts(
                normalize=True,
                dropna=False,
            )
        )

        fitted[column] = {

            "values": frequencies.index.tolist(),

            "probabilities": (
                frequencies.values
                .astype(float)
                .tolist()
            ),

            "dtype": str(
                series.dtype
            ),

            "n_unique": int(
                series.nunique(
                    dropna=False
                )
            ),

            "missing_count": int(
                series.isna().sum()
            ),

            "missing_rate": float(
                series.isna().mean()
            ),
        }

    return fitted


# --------------------------------------------------------------------------------------------------
# 5. Model Containers
# --------------------------------------------------------------------------------------------------

FITTED_BASELINE_MODELS = {}

FIT_RUNTIME_RECORDS = []

BASELINE_METADATA_ARTIFACTS = []

# --------------------------------------------------------------------------------------------------
# 6. Fit Statistical Baselines One Dataset at a Time
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    print()
    print("-" * 100)
    print(
        f"FITTING — {dataset_id}"
    )
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Load training data
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: training data is not available."
        )

    train_df = TRAINING_DATA[
        dataset_id
    ]

    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    target_column = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Training-only validation
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        train_df,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_DATA must contain a pandas DataFrame."
        )

    if not set(generative_columns).issubset(
        train_df.columns
    ):

        missing_columns = [
            column
            for column in generative_columns
            if column not in train_df.columns
        ]

        raise RuntimeError(
            f"{dataset_id}: missing generative column(s): "
            f"{missing_columns}"
        )

    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "is missing from the generative schema."
        )

    if len(train_df) == 0:

        raise RuntimeError(
            f"{dataset_id}: training dataset is empty."
        )

    FITTED_BASELINE_MODELS[
        dataset_id
    ] = {}

    # ----------------------------------------------------------------------------------------------
    # Fit each baseline
    # ----------------------------------------------------------------------------------------------

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        if baseline_name not in BASELINE_CONFIG:

            raise RuntimeError(
                f"{dataset_id}: configuration missing for "
                f"baseline '{baseline_name}'."
            )

        seed = get_baseline_seed(
            dataset_index,
            baseline_index,
        )

        seed_everything(
            seed
        )

        start_time = time.perf_counter()

        # ------------------------------------------------------------------------------------------
        # Independent Marginal
        # ------------------------------------------------------------------------------------------

        if baseline_name == "independent_marginal":

            model = fit_independent_marginal(
                dataframe=train_df[
                    generative_columns
                ],
                columns=generative_columns,
            )

            FITTED_BASELINE_MODELS[
                dataset_id
            ][baseline_name] = model

        # ------------------------------------------------------------------------------------------
        # Gaussian Copula
        # ------------------------------------------------------------------------------------------

        elif baseline_name == "gaussian_copula":

            model_input = (
                train_df[
                    generative_columns
                ]
                .copy()
            )

            # --------------------------------------------------------------------------------------
            # Metadata detection uses TRAINING DATA ONLY.
            # --------------------------------------------------------------------------------------

            metadata = (
                Metadata.detect_from_dataframe(
                    data=model_input,
                    table_name=dataset_id,
                    infer_keys=None,
                )
            )

            metadata.validate()

            # --------------------------------------------------------------------------------------
            # Persist SDV metadata for reproducibility.
            # --------------------------------------------------------------------------------------

            metadata_path = (
                NB04_METADATA_ROOT
                / f"{dataset_id}_gaussian_copula_metadata.json"
            )

            metadata.save_to_json(
                filepath=str(
                    metadata_path
                )
            )

            if not metadata_path.exists():

                raise RuntimeError(
                    f"{dataset_id}: SDV metadata was not persisted."
                )

            if metadata_path.stat().st_size == 0:

                raise RuntimeError(
                    f"{dataset_id}: persisted SDV metadata file is empty."
                )

            BASELINE_METADATA_ARTIFACTS.append(
                {
                    "dataset_id": dataset_id,
                    "baseline": baseline_name,
                    "metadata_path": str(
                        metadata_path
                    ),
                    "metadata_size_bytes": int(
                        metadata_path.stat().st_size
                    ),
                }
            )

            # --------------------------------------------------------------------------------------
            # Configure Gaussian Copula exactly according to Section 6.
            # --------------------------------------------------------------------------------------

            synthesizer = (
                GaussianCopulaSynthesizer(
                    metadata,
                    enforce_min_max_values=(
                        BASELINE_CONFIG[
                            "gaussian_copula"
                        ][
                            "enforce_min_max_values"
                        ]
                    ),
                    enforce_rounding=(
                        BASELINE_CONFIG[
                            "gaussian_copula"
                        ][
                            "enforce_rounding"
                        ]
                    ),
                    default_distribution=(
                        BASELINE_CONFIG[
                            "gaussian_copula"
                        ][
                            "default_distribution"
                        ]
                    ),
                )
            )

            # --------------------------------------------------------------------------------------
            # Fit on NATIVE TRAINING DATA ONLY.
            # --------------------------------------------------------------------------------------

            synthesizer.fit(
                model_input
            )

            FITTED_BASELINE_MODELS[
                dataset_id
            ][baseline_name] = {

                "synthesizer": synthesizer,

                "metadata": metadata,

                "metadata_path": str(
                    metadata_path
                ),
            }

        else:

            raise RuntimeError(
                f"Unknown statistical baseline: "
                f"{baseline_name}"
            )

        # ------------------------------------------------------------------------------------------
        # Runtime recording
        # ------------------------------------------------------------------------------------------

        elapsed = (
            time.perf_counter()
            - start_time
        )

        FIT_RUNTIME_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "seed": int(seed),
                "fit_runtime_seconds": float(
                    elapsed
                ),
                "training_rows": int(
                    len(train_df)
                ),
                "training_columns": int(
                    len(generative_columns)
                ),
                "target_column": target_column,
                "fit_data": "native_train_only",
            }
        )

        print(
            f"✓ {baseline_name:<24} | "
            f"rows={len(train_df):,} | "
            f"generative_columns={len(generative_columns)} | "
            f"fit_runtime={elapsed:.3f}s"
        )

        if baseline_name == "gaussian_copula":

            print(
                f"  Metadata saved : {metadata_path}"
            )

        gc.collect()

# --------------------------------------------------------------------------------------------------
# 7. Final Fit Coverage Validation
# --------------------------------------------------------------------------------------------------

expected_fit_count = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

actual_fit_count = sum(
    len(
        FITTED_BASELINE_MODELS[
            dataset_id
        ]
    )
    for dataset_id in DATASET_IDS
)

if actual_fit_count != expected_fit_count:

    raise RuntimeError(
        f"Expected {expected_fit_count} fitted baseline models, "
        f"but found {actual_fit_count}."
    )

if len(FIT_RUNTIME_RECORDS) != expected_fit_count:

    raise RuntimeError(
        f"Expected {expected_fit_count} runtime records, "
        f"but found {len(FIT_RUNTIME_RECORDS)}."
    )

# --------------------------------------------------------------------------------------------------
# 8. Metadata Artifact Validation
# --------------------------------------------------------------------------------------------------

expected_metadata_count = len(DATASET_IDS)

if len(BASELINE_METADATA_ARTIFACTS) != expected_metadata_count:

    raise RuntimeError(
        f"Expected {expected_metadata_count} Gaussian Copula metadata artifacts, "
        f"but found {len(BASELINE_METADATA_ARTIFACTS)}."
    )

for artifact in BASELINE_METADATA_ARTIFACTS:

    metadata_path = Path(
        artifact["metadata_path"]
    )

    if not metadata_path.exists():

        raise RuntimeError(
            f"Missing metadata artifact: {metadata_path}"
        )

    if metadata_path.stat().st_size <= 0:

        raise RuntimeError(
            f"Empty metadata artifact: {metadata_path}"
        )

# --------------------------------------------------------------------------------------------------
# 9. Final Model Registry Validation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    fitted_baselines = set(
        FITTED_BASELINE_MODELS[
            dataset_id
        ].keys()
    )

    expected_baselines = set(
        BASELINE_METHODS
    )

    if fitted_baselines != expected_baselines:

        raise RuntimeError(
            f"{dataset_id}: fitted baseline registry mismatch. "
            f"Expected {expected_baselines}, "
            f"found {fitted_baselines}."
        )

# --------------------------------------------------------------------------------------------------
# 10. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 8 — FITTING SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets fitted        : {len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset  : {len(BASELINE_METHODS)}"
)

print(
    f"✓ Total fitted models    : {actual_fit_count}"
)

print(
    f"✓ Runtime records        : {len(FIT_RUNTIME_RECORDS)}"
)

print(
    f"✓ Metadata artifacts     : {len(BASELINE_METADATA_ARTIFACTS)}"
)

print(
    "✓ Training-only fitting  : PASS"
)

print(
    "✓ Model coverage         : PASS"
)

print(
    "✓ Metadata persistence   : PASS"
)

print(
    "✓ Fit registry integrity : PASS"
)

print()
print("✓ All statistical baseline distributions fitted.")

SECTION 8 — FIT BASELINE DISTRIBUTIONS
✓ SDV version    : 1.38.3
✓ Python version : 3.13.15
✓ Baseline artifact root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines
✓ Metadata artifact root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/metadata

----------------------------------------------------------------------------------------------------
FITTING — adult_income
----------------------------------------------------------------------------------------------------
✓ independent_marginal     | rows=34,189 | generative_columns=15 | fit_runtime=0.312s
✓ gaussian_copula          | rows=34,189 | generative_columns=15 | fit_runtime=7.607s
  Metadata saved : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/metadata/adult_income_gaussian_copula_metadata.json

----------------------------------------------------------------------------------------------------
FITTING — bank_marketing
------------------

In [40]:
# ==================================================================================================
# 9. GENERATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 9 — GENERATE SYNTHETIC DATA")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Runtime Containers
# --------------------------------------------------------------------------------------------------

SYNTHETIC_DATA = {}

GENERATION_RUNTIME_RECORDS = []

# --------------------------------------------------------------------------------------------------
# 2. Generate Synthetic Data Dataset-by-Dataset
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    print()
    print("-" * 100)
    print(
        f"GENERATING — {dataset_id}"
    )
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Retrieve authoritative training data
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: training data is not available."
        )

    train_df = TRAINING_DATA[
        dataset_id
    ]

    if not isinstance(
        train_df,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: training data is not a pandas DataFrame."
        )

    if len(train_df) == 0:

        raise RuntimeError(
            f"{dataset_id}: training dataset is empty."
        )

    # ----------------------------------------------------------------------------------------------
    # Training row count — authoritative source
    # ----------------------------------------------------------------------------------------------

    training_rows = int(
        len(train_df)
    )

    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    target_column = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Validate frozen generative schema
    # ----------------------------------------------------------------------------------------------

    if not set(generative_columns).issubset(
        train_df.columns
    ):

        missing_columns = [
            column
            for column in generative_columns
            if column not in train_df.columns
        ]

        raise RuntimeError(
            f"{dataset_id}: missing generative column(s): "
            f"{missing_columns}"
        )

    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            "is not present in the generative schema."
        )

    SYNTHETIC_DATA[
        dataset_id
    ] = {}

    # ----------------------------------------------------------------------------------------------
    # Generate each baseline
    # ----------------------------------------------------------------------------------------------

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        # ------------------------------------------------------------------------------------------
        # Validate configuration
        # ------------------------------------------------------------------------------------------

        if baseline_name not in BASELINE_CONFIG:

            raise RuntimeError(
                f"{dataset_id}: configuration missing for "
                f"baseline '{baseline_name}'."
            )

        configured_sample_size = (
            BASELINE_CONFIG[
                baseline_name
            ][
                "sample_size"
            ]
        )

        if configured_sample_size != "training_rows":

            raise RuntimeError(
                f"{dataset_id} / {baseline_name}: "
                "sample-size configuration must be 'training_rows'."
            )

        # ------------------------------------------------------------------------------------------
        # Deterministic seed
        # ------------------------------------------------------------------------------------------

        seed = get_baseline_seed(
            dataset_index,
            baseline_index,
        )

        seed_everything(
            seed
        )

        start_time = time.perf_counter()

        # ------------------------------------------------------------------------------------------
        # Independent Marginal
        # ------------------------------------------------------------------------------------------

        if baseline_name == "independent_marginal":

            model = (
                FITTED_BASELINE_MODELS[
                    dataset_id
                ][baseline_name]
            )

            rng = np.random.default_rng(
                seed
            )

            synthetic_columns = {}

            for column in generative_columns:

                if column not in model:

                    raise RuntimeError(
                        f"{dataset_id}: Independent Marginal model "
                        f"is missing column '{column}'."
                    )

                values = np.asarray(
                    model[column]["values"],
                    dtype=object,
                )

                probabilities = np.asarray(
                    model[column]["probabilities"],
                    dtype=float,
                )

                if len(values) == 0:

                    raise RuntimeError(
                        f"{dataset_id}: no empirical values available "
                        f"for column '{column}'."
                    )

                if len(values) != len(probabilities):

                    raise RuntimeError(
                        f"{dataset_id}: value/probability length mismatch "
                        f"for column '{column}'."
                    )

                probability_sum = (
                    probabilities.sum()
                )

                if not np.isfinite(
                    probability_sum
                ) or probability_sum <= 0:

                    raise RuntimeError(
                        f"{dataset_id}: invalid probability distribution "
                        f"for column '{column}'."
                    )

                probabilities = (
                    probabilities
                    / probability_sum
                )

                sampled = rng.choice(
                    values,
                    size=training_rows,
                    replace=True,
                    p=probabilities,
                )

                sampled_series = pd.Series(
                    sampled,
                    dtype=object,
                )

                sampled_series = (
                    sampled_series
                    .replace(
                        "__NB04_MISSING__",
                        np.nan,
                    )
                )

                synthetic_columns[
                    column
                ] = sampled_series

            synthetic_df = pd.DataFrame(
                synthetic_columns,
                columns=generative_columns,
            )

        # ------------------------------------------------------------------------------------------
        # Gaussian Copula
        # ------------------------------------------------------------------------------------------

        elif baseline_name == "gaussian_copula":

            model_record = (
                FITTED_BASELINE_MODELS[
                    dataset_id
                ][baseline_name]
            )

            synthesizer = (
                model_record[
                    "synthesizer"
                ]
            )

            # --------------------------------------------------------------------------------------
            # Generate the exact training-row sample size.
            # --------------------------------------------------------------------------------------

            try:

                synthetic_df = (
                    synthesizer.sample(
                        num_rows=training_rows,
                        randomize_samples=True,
                    )
                )

            except TypeError:

                # Compatibility fallback for SDV versions whose sample()
                # does not expose randomize_samples.
                synthetic_df = (
                    synthesizer.sample(
                        num_rows=training_rows
                    )
                )

            # --------------------------------------------------------------------------------------
            # Preserve only the frozen generative schema.
            # --------------------------------------------------------------------------------------

            missing_columns = [
                column
                for column in generative_columns
                if column not in synthetic_df.columns
            ]

            if missing_columns:

                raise RuntimeError(
                    f"{dataset_id}: Gaussian Copula generated data is "
                    f"missing column(s): {missing_columns}"
                )

            synthetic_df = (
                synthetic_df[
                    generative_columns
                ]
                .copy()
            )

        else:

            raise RuntimeError(
                f"Unknown statistical baseline: "
                f"{baseline_name}"
            )

        # ------------------------------------------------------------------------------------------
        # Validate generated object
        # ------------------------------------------------------------------------------------------

        if not isinstance(
            synthetic_df,
            pd.DataFrame,
        ):

            raise RuntimeError(
                f"{dataset_id} / {baseline_name}: "
                "synthetic output is not a pandas DataFrame."
            )

        # ------------------------------------------------------------------------------------------
        # Validate row count
        # ------------------------------------------------------------------------------------------

        rows_generated = int(
            len(synthetic_df)
        )

        if rows_generated != training_rows:

            raise RuntimeError(
                f"{dataset_id} / {baseline_name}: "
                f"expected {training_rows:,} rows but generated "
                f"{rows_generated:,}."
            )

        # ------------------------------------------------------------------------------------------
        # Validate exact generated schema
        # ------------------------------------------------------------------------------------------

        if list(
            synthetic_df.columns
        ) != list(
            generative_columns
        ):

            raise RuntimeError(
                f"{dataset_id} / {baseline_name}: "
                "generated schema does not exactly match the frozen "
                "generative schema."
            )

        # ------------------------------------------------------------------------------------------
        # Explicit provenance exclusion
        # ------------------------------------------------------------------------------------------

        provenance_column = (
            TRAINING_PROVENANCE_COLUMNS[
                dataset_id
            ]
        )

        if provenance_column is not None:

            if provenance_column in synthetic_df.columns:

                raise RuntimeError(
                    f"{dataset_id} / {baseline_name}: "
                    f"provenance column '{provenance_column}' "
                    "must not be generated."
                )

        # ------------------------------------------------------------------------------------------
        # Explicit identifier exclusion
        # ------------------------------------------------------------------------------------------

        identifier_columns = (
            TRAINING_IDENTIFIER_COLUMNS[
                dataset_id
            ]
        )

        leaked_identifiers = [
            column
            for column in identifier_columns
            if column in synthetic_df.columns
        ]

        if leaked_identifiers:

            raise RuntimeError(
                f"{dataset_id} / {baseline_name}: "
                f"identifier column(s) leaked into synthetic data: "
                f"{leaked_identifiers}"
            )

        # ------------------------------------------------------------------------------------------
        # Runtime
        # ------------------------------------------------------------------------------------------

        elapsed = (
            time.perf_counter()
            - start_time
        )

        SYNTHETIC_DATA[
            dataset_id
        ][baseline_name] = synthetic_df

        GENERATION_RUNTIME_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "seed": int(seed),
                "rows_requested": training_rows,
                "rows_generated": rows_generated,
                "generative_columns": len(
                    generative_columns
                ),
                "target_column": target_column,
                "generation_runtime_seconds": float(
                    elapsed
                ),
            }
        )

        print(
            f"✓ {baseline_name:<24} | "
            f"rows={rows_generated:>8,} | "
            f"columns={len(generative_columns):>4} | "
            f"runtime={elapsed:.3f}s"
        )

        gc.collect()

# --------------------------------------------------------------------------------------------------
# 3. Final Generation Coverage Validation
# --------------------------------------------------------------------------------------------------

expected_generation_count = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

actual_generation_count = sum(
    len(
        SYNTHETIC_DATA[
            dataset_id
        ]
    )
    for dataset_id in DATASET_IDS
)

if actual_generation_count != expected_generation_count:

    raise RuntimeError(
        f"Expected {expected_generation_count} generated datasets, "
        f"but found {actual_generation_count}."
    )

if len(GENERATION_RUNTIME_RECORDS) != expected_generation_count:

    raise RuntimeError(
        f"Expected {expected_generation_count} generation runtime records, "
        f"but found {len(GENERATION_RUNTIME_RECORDS)}."
    )

# --------------------------------------------------------------------------------------------------
# 4. Final Synthetic Dataset Validation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    expected_rows = int(
        len(
            TRAINING_DATA[
                dataset_id
            ]
        )
    )

    expected_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    for baseline_name in BASELINE_METHODS:

        synthetic_df = (
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        if len(synthetic_df) != expected_rows:

            raise RuntimeError(
                f"{dataset_id} / {baseline_name}: "
                "final row-count validation failed."
            )

        if list(
            synthetic_df.columns
        ) != expected_columns:

            raise RuntimeError(
                f"{dataset_id} / {baseline_name}: "
                "final schema validation failed."
            )

# --------------------------------------------------------------------------------------------------
# 5. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 9 — GENERATION SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets generated       : {len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset    : {len(BASELINE_METHODS)}"
)

print(
    f"✓ Total synthetic datasets : {actual_generation_count}"
)

print(
    f"✓ Runtime records          : {len(GENERATION_RUNTIME_RECORDS)}"
)

print(
    "✓ Training-sized generation: PASS"
)

print(
    "✓ Exact generative schema  : PASS"
)

print(
    "✓ Provenance exclusion     : PASS"
)

print(
    "✓ Identifier exclusion     : PASS"
)

print(
    "✓ Generation coverage      : PASS"
)

print()
print("✓ Synthetic datasets generated successfully.")

SECTION 9 — GENERATE SYNTHETIC DATA

----------------------------------------------------------------------------------------------------
GENERATING — adult_income
----------------------------------------------------------------------------------------------------
✓ independent_marginal     | rows=  34,189 | columns=  15 | runtime=0.226s
✓ gaussian_copula          | rows=  34,189 | columns=  15 | runtime=0.965s

----------------------------------------------------------------------------------------------------
GENERATING — bank_marketing
----------------------------------------------------------------------------------------------------
✓ independent_marginal     | rows=  31,647 | columns=  17 | runtime=0.160s
✓ gaussian_copula          | rows=  31,647 | columns=  17 | runtime=0.894s

----------------------------------------------------------------------------------------------------
GENERATING — diabetes_130us
--------------------------------------------------------------------------

In [41]:
# ==================================================================================================
# 10. VALIDATE SYNTHETIC SCHEMA
# ==================================================================================================

print("=" * 100)
print("SECTION 10 — VALIDATE SYNTHETIC SCHEMA")
print("=" * 100)

SYNTHETIC_SCHEMA_VALIDATION = []

for dataset_id in DATASET_IDS:

    expected_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    for baseline_name in BASELINE_METHODS:

        synthetic_df = (
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        # ------------------------------------------------------------------------------------------
        # Exact column order
        # ------------------------------------------------------------------------------------------

        if list(
            synthetic_df.columns
        ) != list(
            expected_columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "synthetic column schema mismatch."
            )

        # ------------------------------------------------------------------------------------------
        # Target retained
        # ------------------------------------------------------------------------------------------

        if target not in synthetic_df.columns:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "target column missing."
            )

        # ------------------------------------------------------------------------------------------
        # Provenance excluded
        # ------------------------------------------------------------------------------------------

        if provenance in synthetic_df.columns:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "provenance leakage detected."
            )

        # ------------------------------------------------------------------------------------------
        # Identifiers excluded
        # ------------------------------------------------------------------------------------------

        identifier_leakage = (
            set(identifiers)
            .intersection(
                synthetic_df.columns
            )
        )

        if identifier_leakage:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"identifier leakage: "
                f"{sorted(identifier_leakage)}"
            )

        SYNTHETIC_SCHEMA_VALIDATION.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "schema_pass": True,
                "columns": len(
                    synthetic_df.columns
                ),
                "target_present": True,
                "provenance_excluded": True,
                "identifiers_excluded": True,
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"schema PASS"
        )

SYNTHETIC_SCHEMA_VALIDATION_DF = pd.DataFrame(
    SYNTHETIC_SCHEMA_VALIDATION
)

print()
print("✓ Synthetic schema validation : PASS")

SECTION 10 — VALIDATE SYNTHETIC SCHEMA
✓ adult_income         | independent_marginal     | schema PASS
✓ adult_income         | gaussian_copula          | schema PASS
✓ bank_marketing       | independent_marginal     | schema PASS
✓ bank_marketing       | gaussian_copula          | schema PASS
✓ diabetes_130us       | independent_marginal     | schema PASS
✓ diabetes_130us       | gaussian_copula          | schema PASS

✓ Synthetic schema validation : PASS


In [43]:
# ==================================================================================================
# 11. VALIDATE SAMPLE SIZE
# ==================================================================================================

print("=" * 100)
print("SECTION 11 — VALIDATE SAMPLE SIZE")
print("=" * 100)

SAMPLE_SIZE_VALIDATION = []

for dataset_id in DATASET_IDS:

    # --------------------------------------------------------------------------------------------------
    # Authoritative training-row count
    # --------------------------------------------------------------------------------------------------

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: training data is not available."
        )

    training_df = TRAINING_DATA[
        dataset_id
    ]

    if not isinstance(
        training_df,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: training data must be a pandas DataFrame."
        )

    expected_rows = int(
        len(training_df)
    )

    if expected_rows <= 0:

        raise RuntimeError(
            f"{dataset_id}: invalid training-row count: "
            f"{expected_rows}"
        )

    for baseline_name in BASELINE_METHODS:

        # ----------------------------------------------------------------------------------------------
        # Verify synthetic dataset exists
        # ----------------------------------------------------------------------------------------------

        if dataset_id not in SYNTHETIC_DATA:

            raise RuntimeError(
                f"{dataset_id}: synthetic data container is missing."
            )

        if baseline_name not in SYNTHETIC_DATA[
            dataset_id
        ]:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "synthetic dataset is missing."
            )

        synthetic_df = (
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        if not isinstance(
            synthetic_df,
            pd.DataFrame,
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "synthetic data must be a pandas DataFrame."
            )

        # ----------------------------------------------------------------------------------------------
        # Actual sample size
        # ----------------------------------------------------------------------------------------------

        actual_rows = int(
            len(synthetic_df)
        )

        # ----------------------------------------------------------------------------------------------
        # Exact sample-size validation
        # ----------------------------------------------------------------------------------------------

        if actual_rows != expected_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"sample-size mismatch.\n"
                f"Expected: {expected_rows:,}\n"
                f"Found   : {actual_rows:,}"
            )

        SAMPLE_SIZE_VALIDATION.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "expected_rows": expected_rows,
                "actual_rows": actual_rows,
                "difference": actual_rows - expected_rows,
                "sample_size_pass": True,
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"{actual_rows:>8,} rows PASS"
        )

# --------------------------------------------------------------------------------------------------
# Final Validation DataFrame
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_VALIDATION_DF = pd.DataFrame(
    SAMPLE_SIZE_VALIDATION
)

# --------------------------------------------------------------------------------------------------
# Final Integrity Gate
# --------------------------------------------------------------------------------------------------

expected_validation_records = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

if len(
    SAMPLE_SIZE_VALIDATION_DF
) != expected_validation_records:

    raise RuntimeError(
        "Unexpected number of sample-size validation records. "
        f"Expected {expected_validation_records}, "
        f"found {len(SAMPLE_SIZE_VALIDATION_DF)}."
    )

if not SAMPLE_SIZE_VALIDATION_DF[
    "sample_size_pass"
].all():

    raise RuntimeError(
        "One or more sample-size validations failed."
    )

# --------------------------------------------------------------------------------------------------
# Summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Validation records : "
    f"{len(SAMPLE_SIZE_VALIDATION_DF)}"
)

print(
    f"✓ Datasets validated : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines validated: "
    f"{len(BASELINE_METHODS)}"
)

print(
    "✓ Training-row reference : TRAINING_DATA"
)

print(
    "✓ Exact sample-size validation : PASS"
)

print(
    "✓ Synthetic sample-size validation : PASS"
)

SECTION 11 — VALIDATE SAMPLE SIZE
✓ adult_income         | independent_marginal     |   34,189 rows PASS
✓ adult_income         | gaussian_copula          |   34,189 rows PASS
✓ bank_marketing       | independent_marginal     |   31,647 rows PASS
✓ bank_marketing       | gaussian_copula          |   31,647 rows PASS
✓ diabetes_130us       | independent_marginal     |   71,236 rows PASS
✓ diabetes_130us       | gaussian_copula          |   71,236 rows PASS

----------------------------------------------------------------------------------------------------
✓ Validation records : 6
✓ Datasets validated : 3
✓ Baselines validated: 2
✓ Training-row reference : TRAINING_DATA
✓ Exact sample-size validation : PASS
✓ Synthetic sample-size validation : PASS


In [45]:
# ==================================================================================================
# 12. RECORD RUNTIME
# ==================================================================================================

print("=" * 100)
print("SECTION 12 — RECORD RUNTIME")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Convert Runtime Records to DataFrames
# --------------------------------------------------------------------------------------------------

FIT_RUNTIME_DF = pd.DataFrame(
    FIT_RUNTIME_RECORDS
)

GENERATION_RUNTIME_DF = pd.DataFrame(
    GENERATION_RUNTIME_RECORDS
)

# --------------------------------------------------------------------------------------------------
# 2. Validate Runtime Record Counts
# --------------------------------------------------------------------------------------------------

EXPECTED_RUNTIME_RECORDS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

if len(FIT_RUNTIME_DF) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNTIME_RECORDS} fit runtime records, "
        f"found {len(FIT_RUNTIME_DF)}."
    )

if len(GENERATION_RUNTIME_DF) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNTIME_RECORDS} generation runtime records, "
        f"found {len(GENERATION_RUNTIME_DF)}."
    )

# --------------------------------------------------------------------------------------------------
# 3. Validate Runtime Record Uniqueness
# --------------------------------------------------------------------------------------------------

RUNTIME_KEYS = [
    "dataset_id",
    "baseline",
    "seed",
]

if FIT_RUNTIME_DF.duplicated(
    subset=RUNTIME_KEYS
).any():

    raise RuntimeError(
        "Duplicate fit runtime records detected."
    )

if GENERATION_RUNTIME_DF.duplicated(
    subset=RUNTIME_KEYS
).any():

    raise RuntimeError(
        "Duplicate generation runtime records detected."
    )

# --------------------------------------------------------------------------------------------------
# 4. Merge Fit and Generation Runtime Records
# --------------------------------------------------------------------------------------------------

RUNTIME_DF = (
    FIT_RUNTIME_DF
    .merge(
        GENERATION_RUNTIME_DF,
        on=RUNTIME_KEYS,
        how="inner",
        validate="one_to_one",
    )
)

if len(RUNTIME_DF) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNTIME_RECORDS} merged runtime records, "
        f"found {len(RUNTIME_DF)}."
    )

# --------------------------------------------------------------------------------------------------
# 5. Validate Expected Dataset/Baseline Coverage
# --------------------------------------------------------------------------------------------------

expected_runtime_keys = {
    (
        dataset_id,
        baseline_name,
        get_baseline_seed(
            dataset_index,
            baseline_index,
        ),
    )
    for dataset_index, dataset_id in enumerate(DATASET_IDS)
    for baseline_index, baseline_name in enumerate(BASELINE_METHODS)
}

actual_runtime_keys = {
    (
        row["dataset_id"],
        row["baseline"],
        int(row["seed"]),
    )
    for _, row in RUNTIME_DF.iterrows()
}

if actual_runtime_keys != expected_runtime_keys:

    missing_keys = (
        expected_runtime_keys
        - actual_runtime_keys
    )

    unexpected_keys = (
        actual_runtime_keys
        - expected_runtime_keys
    )

    raise RuntimeError(
        "Runtime coverage mismatch.\n"
        f"Missing: {sorted(missing_keys)}\n"
        f"Unexpected: {sorted(unexpected_keys)}"
    )

# --------------------------------------------------------------------------------------------------
# 6. Validate Runtime Columns
# --------------------------------------------------------------------------------------------------

required_runtime_columns = [
    "dataset_id",
    "baseline",
    "seed",
    "rows_requested",
    "rows_generated",
    "fit_runtime_seconds",
    "generation_runtime_seconds",
]

missing_runtime_columns = [
    column
    for column in required_runtime_columns
    if column not in RUNTIME_DF.columns
]

if missing_runtime_columns:

    raise RuntimeError(
        f"Missing runtime column(s): {missing_runtime_columns}"
    )

# --------------------------------------------------------------------------------------------------
# 7. Validate Runtime Values
# --------------------------------------------------------------------------------------------------

runtime_numeric_columns = [
    "rows_requested",
    "rows_generated",
    "fit_runtime_seconds",
    "generation_runtime_seconds",
]

for column in runtime_numeric_columns:

    RUNTIME_DF[column] = pd.to_numeric(
        RUNTIME_DF[column],
        errors="raise",
    )

for column in [
    "fit_runtime_seconds",
    "generation_runtime_seconds",
]:

    if not np.isfinite(
        RUNTIME_DF[column]
    ).all():

        raise RuntimeError(
            f"Non-finite runtime value detected in '{column}'."
        )

    if (
        RUNTIME_DF[column] < 0
    ).any():

        raise RuntimeError(
            f"Negative runtime value detected in '{column}'."
        )

# --------------------------------------------------------------------------------------------------
# 8. Validate Requested vs Generated Rows
# --------------------------------------------------------------------------------------------------

if not (
    RUNTIME_DF["rows_requested"]
    == RUNTIME_DF["rows_generated"]
).all():

    raise RuntimeError(
        "One or more runtime records have inconsistent "
        "requested/generated row counts."
    )

# --------------------------------------------------------------------------------------------------
# 9. Calculate Total Runtime
# --------------------------------------------------------------------------------------------------

RUNTIME_DF[
    "total_runtime_seconds"
] = (
    RUNTIME_DF[
        "fit_runtime_seconds"
    ]
    +
    RUNTIME_DF[
        "generation_runtime_seconds"
    ]
)

# --------------------------------------------------------------------------------------------------
# 10. Validate Total Runtime
# --------------------------------------------------------------------------------------------------

expected_total_runtime = (
    RUNTIME_DF[
        "fit_runtime_seconds"
    ]
    +
    RUNTIME_DF[
        "generation_runtime_seconds"
    ]
)

if not np.allclose(
    RUNTIME_DF[
        "total_runtime_seconds"
    ],
    expected_total_runtime,
    rtol=0,
    atol=1e-12,
):

    raise RuntimeError(
        "Total runtime calculation validation failed."
    )

# --------------------------------------------------------------------------------------------------
# 11. Select Final Runtime Schema
# --------------------------------------------------------------------------------------------------

RUNTIME_DF = RUNTIME_DF[
    [
        "dataset_id",
        "baseline",
        "seed",
        "rows_requested",
        "rows_generated",
        "fit_runtime_seconds",
        "generation_runtime_seconds",
        "total_runtime_seconds",
    ]
].copy()

# --------------------------------------------------------------------------------------------------
# 12. Display Runtime Records
# --------------------------------------------------------------------------------------------------

print()

print(
    RUNTIME_DF.to_string(
        index=False
    )
)

# --------------------------------------------------------------------------------------------------
# 13. Final Runtime Integrity Gate
# --------------------------------------------------------------------------------------------------

if len(RUNTIME_DF) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        "Final runtime record count validation failed."
    )

if RUNTIME_DF[
    "total_runtime_seconds"
].isna().any():

    raise RuntimeError(
        "Missing total runtime value detected."
    )

print()
print("-" * 100)

print(
    f"✓ Fit runtime records       : {len(FIT_RUNTIME_DF)}"
)

print(
    f"✓ Generation runtime records: {len(GENERATION_RUNTIME_DF)}"
)

print(
    f"✓ Merged runtime records    : {len(RUNTIME_DF)}"
)

print(
    "✓ Runtime key uniqueness    : PASS"
)

print(
    "✓ Runtime coverage          : PASS"
)

print(
    "✓ Runtime value validation  : PASS"
)

print(
    "✓ Row-count consistency     : PASS"
)

print(
    "✓ Total-runtime calculation : PASS"
)

print(
    "✓ Runtime records created."
)

SECTION 12 — RECORD RUNTIME

    dataset_id             baseline  seed  rows_requested  rows_generated  fit_runtime_seconds  generation_runtime_seconds  total_runtime_seconds
  adult_income independent_marginal  3125           34189           34189             0.312164                    0.226107               0.538271
  adult_income      gaussian_copula  3225           34189           34189             7.606904                    0.965154               8.572058
bank_marketing independent_marginal  4125           31647           31647             0.233809                    0.160306               0.394115
bank_marketing      gaussian_copula  4225           31647           31647             5.223438                    0.893786               6.117224
diabetes_130us independent_marginal  5125           71236           71236             0.873311                    0.816736               1.690047
diabetes_130us      gaussian_copula  5225           71236           71236            40.689073 

In [47]:
# ==================================================================================================
# 13. SAVE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 13 — SAVE SYNTHETIC DATA")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Define Canonical Notebook 04 Paths
# --------------------------------------------------------------------------------------------------

NB04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

NB04_SYNTHETIC_ROOT = (
    NB04_ROOT
    / "synthetic"
)

NB04_VALIDATION_ROOT = (
    NB04_ROOT
    / "validation"
)

for directory in [
    NB04_ROOT,
    NB04_SYNTHETIC_ROOT,
    NB04_VALIDATION_ROOT,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# --------------------------------------------------------------------------------------------------
# 2. Reset Artifact Records
# --------------------------------------------------------------------------------------------------

SYNTHETIC_ARTIFACT_RECORDS = []

# --------------------------------------------------------------------------------------------------
# 3. Persist Synthetic Datasets
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_root = (
        NB04_SYNTHETIC_ROOT
        / dataset_id
    )

    dataset_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    expected_generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    expected_identifier_columns = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    for baseline_name in BASELINE_METHODS:

        # ------------------------------------------------------------------------------------------
        # Retrieve In-Memory Synthetic Dataset
        # ------------------------------------------------------------------------------------------

        if dataset_id not in SYNTHETIC_DATA:

            raise RuntimeError(
                f"{dataset_id}: synthetic dataset container is missing."
            )

        if baseline_name not in SYNTHETIC_DATA[dataset_id]:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: synthetic dataset is missing."
            )

        synthetic_df = (
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        if not isinstance(
            synthetic_df,
            pd.DataFrame,
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: synthetic object is not a DataFrame."
            )

        # ------------------------------------------------------------------------------------------
        # Validate In-Memory Schema Before Persistence
        # ------------------------------------------------------------------------------------------

        if list(
            synthetic_df.columns
        ) != list(
            expected_generative_columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "synthetic schema does not match frozen generative schema."
            )

        if (
            "__original_row_id__"
            in synthetic_df.columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: provenance column detected."
            )

        unexpected_identifiers = [
            column
            for column in expected_identifier_columns
            if column in synthetic_df.columns
        ]

        if unexpected_identifiers:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"explicit identifier column(s) detected: "
                f"{unexpected_identifiers}"
            )

        # ------------------------------------------------------------------------------------------
        # Define Output Path
        # ------------------------------------------------------------------------------------------

        output_path = (
            dataset_root
            / f"{baseline_name}.csv"
        )

        # ------------------------------------------------------------------------------------------
        # Save Synthetic Dataset
        # ------------------------------------------------------------------------------------------

        synthetic_df.to_csv(
            output_path,
            index=False,
        )

        if not output_path.exists():

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: output file was not created."
            )

        file_size_bytes = (
            output_path.stat().st_size
        )

        if file_size_bytes <= 0:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: output file is empty."
            )

        # ------------------------------------------------------------------------------------------
        # Calculate SHA-256
        # ------------------------------------------------------------------------------------------

        file_hash = calculate_sha256(
            output_path
        )

        if not file_hash:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: SHA-256 calculation failed."
            )

        # ------------------------------------------------------------------------------------------
        # Reload Persisted Dataset
        # ------------------------------------------------------------------------------------------

        reloaded_df = pd.read_csv(
            output_path
        )

        # ------------------------------------------------------------------------------------------
        # Validate Persisted Row Count
        # ------------------------------------------------------------------------------------------

        if len(
            reloaded_df
        ) != len(
            synthetic_df
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "persisted row count does not match in-memory row count."
            )

        # ------------------------------------------------------------------------------------------
        # Validate Persisted Column Count
        # ------------------------------------------------------------------------------------------

        if len(
            reloaded_df.columns
        ) != len(
            synthetic_df.columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "persisted column count does not match in-memory column count."
            )

        # ------------------------------------------------------------------------------------------
        # Validate Persisted Schema
        # ------------------------------------------------------------------------------------------

        if list(
            reloaded_df.columns
        ) != list(
            expected_generative_columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "persisted schema does not match frozen generative schema."
            )

        # ------------------------------------------------------------------------------------------
        # Validate Provenance Exclusion
        # ------------------------------------------------------------------------------------------

        if (
            "__original_row_id__"
            in reloaded_df.columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "provenance column persisted unexpectedly."
            )

        # ------------------------------------------------------------------------------------------
        # Validate Identifier Exclusion
        # ------------------------------------------------------------------------------------------

        persisted_identifiers = [
            column
            for column in expected_identifier_columns
            if column in reloaded_df.columns
        ]

        if persisted_identifiers:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"identifier column(s) persisted unexpectedly: "
                f"{persisted_identifiers}"
            )

        # ------------------------------------------------------------------------------------------
        # Record Artifact Metadata
        # ------------------------------------------------------------------------------------------

        SYNTHETIC_ARTIFACT_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "artifact_type": "synthetic_data",
                "path": str(output_path),
                "relative_path": str(
                    output_path.relative_to(
                        NB04_ROOT
                    )
                ),
                "rows": len(
                    reloaded_df
                ),
                "columns": len(
                    reloaded_df.columns
                ),
                "file_size_bytes": file_size_bytes,
                "sha256": file_hash,
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"{len(reloaded_df):>7} rows | "
            f"{len(reloaded_df.columns):>3} cols | "
            f"{file_size_bytes:>10} bytes | "
            f"persistence verified"
        )

# --------------------------------------------------------------------------------------------------
# 4. Create Artifact Registry
# --------------------------------------------------------------------------------------------------

SYNTHETIC_ARTIFACT_DF = pd.DataFrame(
    SYNTHETIC_ARTIFACT_RECORDS
)

# --------------------------------------------------------------------------------------------------
# 5. Final Artifact Count Validation
# --------------------------------------------------------------------------------------------------

EXPECTED_SYNTHETIC_ARTIFACTS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

if len(
    SYNTHETIC_ARTIFACT_DF
) != EXPECTED_SYNTHETIC_ARTIFACTS:

    raise RuntimeError(
        f"Expected {EXPECTED_SYNTHETIC_ARTIFACTS} synthetic artifacts, "
        f"found {len(SYNTHETIC_ARTIFACT_DF)}."
    )

# --------------------------------------------------------------------------------------------------
# 6. Validate Artifact Registry Uniqueness
# --------------------------------------------------------------------------------------------------

artifact_keys = [
    "dataset_id",
    "baseline",
]

if SYNTHETIC_ARTIFACT_DF.duplicated(
    subset=artifact_keys
).any():

    raise RuntimeError(
        "Duplicate synthetic artifact records detected."
    )

# --------------------------------------------------------------------------------------------------
# 7. Validate Artifact Coverage
# --------------------------------------------------------------------------------------------------

expected_artifact_keys = {
    (
        dataset_id,
        baseline_name,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}

actual_artifact_keys = {
    (
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in SYNTHETIC_ARTIFACT_DF.iterrows()
}

if actual_artifact_keys != expected_artifact_keys:

    raise RuntimeError(
        "Synthetic artifact coverage mismatch."
    )

# --------------------------------------------------------------------------------------------------
# 8. Final Persistence Summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Synthetic datasets saved       : "
    f"{len(SYNTHETIC_ARTIFACT_DF)}"
)

print(
    f"✓ Persistence verification       : PASS"
)

print(
    f"✓ Schema persistence validation  : PASS"
)

print(
    f"✓ Row-count persistence          : PASS"
)

print(
    f"✓ Provenance exclusion           : PASS"
)

print(
    f"✓ Identifier exclusion           : PASS"
)

print(
    f"✓ Artifact coverage              : PASS"
)

print(
    f"✓ Artifact registry uniqueness   : PASS"
)

print(
    "✓ SECTION 13 — SYNTHETIC DATA SAVE: PASS"
)

SECTION 13 — SAVE SYNTHETIC DATA
✓ adult_income         | independent_marginal     |   34189 rows |  15 cols |    3688273 bytes | persistence verified
✓ adult_income         | gaussian_copula          |   34189 rows |  15 cols |    3792889 bytes | persistence verified
✓ bank_marketing       | independent_marginal     |   31647 rows |  17 cols |    2595084 bytes | persistence verified
✓ bank_marketing       | gaussian_copula          |   31647 rows |  17 cols |    2626415 bytes | persistence verified
✓ diabetes_130us       | independent_marginal     |   71236 rows |  48 cols |   11377577 bytes | persistence verified
✓ diabetes_130us       | gaussian_copula          |   71236 rows |  48 cols |   11373182 bytes | persistence verified

----------------------------------------------------------------------------------------------------
✓ Synthetic datasets saved       : 6
✓ Persistence verification       : PASS
✓ Schema persistence validation  : PASS
✓ Row-count persistence          : PASS


In [51]:
# ==================================================================================================
# 14. SAVE BASELINE METADATA
# ==================================================================================================

print("=" * 100)
print("SECTION 14 — SAVE BASELINE METADATA")
print("=" * 100)

from datetime import datetime, timezone

# --------------------------------------------------------------------------------------------------
# 1. Initialize Metadata Registry
# --------------------------------------------------------------------------------------------------

BASELINE_METADATA_RECORDS = []

# Explicitly derive the frozen sampling policy from Notebook 04 Section 6/9.
# Both baselines are generated at the training-set size.
SYNTHETIC_SAMPLE_POLICY = "training_rows"

# --------------------------------------------------------------------------------------------------
# 2. Generate and Persist Metadata for Every Dataset × Baseline
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(DATASET_IDS):

    train_df = TRAINING_DATA[
        dataset_id
    ]

    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    feature_columns = (
        TRAINING_FEATURE_COLUMNS[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Validate Training Data Availability
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        train_df,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: training data is not a DataFrame."
        )

    # ----------------------------------------------------------------------------------------------
    # Determine Numeric and Categorical Columns
    # ----------------------------------------------------------------------------------------------

    numeric_columns = [
        column
        for column in generative_columns
        if pd.api.types.is_numeric_dtype(
            train_df[column]
        )
    ]

    categorical_columns = [
        column
        for column in generative_columns
        if column not in numeric_columns
    ]

    # ----------------------------------------------------------------------------------------------
    # Validate Frozen Generative Schema
    # ----------------------------------------------------------------------------------------------

    if list(
        train_df.columns
    ) != (
        [provenance]
        + list(generative_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: training schema does not match "
            "the frozen Notebook 02 native schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Dataset × Baseline Metadata
    # ----------------------------------------------------------------------------------------------

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        # ------------------------------------------------------------------------------------------
        # Baseline Seed
        # ------------------------------------------------------------------------------------------

        seed = get_baseline_seed(
            dataset_index,
            baseline_index,
        )

        # ------------------------------------------------------------------------------------------
        # Validate Synthetic Dataset
        # ------------------------------------------------------------------------------------------

        synthetic_df = (
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        if not isinstance(
            synthetic_df,
            pd.DataFrame,
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "synthetic data is not a DataFrame."
            )

        # ------------------------------------------------------------------------------------------
        # Validate Runtime Record
        # ------------------------------------------------------------------------------------------

        runtime_records = RUNTIME_DF[
            (
                RUNTIME_DF["dataset_id"]
                == dataset_id
            )
            &
            (
                RUNTIME_DF["baseline"]
                == baseline_name
            )
            &
            (
                RUNTIME_DF["seed"]
                == seed
            )
        ].to_dict(
            orient="records"
        )

        if len(
            runtime_records
        ) != 1:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"expected exactly one runtime record, "
                f"found {len(runtime_records)}."
            )

        # ------------------------------------------------------------------------------------------
        # Construct Metadata
        # ------------------------------------------------------------------------------------------

        baseline_definition = (
            BASELINE_DEFINITIONS[
                baseline_name
            ]
        )

        metadata = {

            "notebook": {
                "id": NOTEBOOK_ID,
                "name": NOTEBOOK_NAME,
                "version": NOTEBOOK_VERSION,
            },

            "dataset_id": dataset_id,

            "baseline": {
                "id": baseline_name,
                "name": baseline_definition[
                    "name"
                ],
                "family": baseline_definition[
                    "family"
                ],
                "definition": baseline_definition[
                    "principle"
                ],
                "dependency_model": baseline_definition[
                    "dependency_model"
                ],
                "formal_dp": False,
            },

            "data_source": {
                "notebook": "02",
                "layer": "native",
                "fit_split": "train",
                "validation_used": False,
                "test_used": False,
            },

            "schema": {
                "generative_columns": list(
                    generative_columns
                ),
                "feature_columns": list(
                    feature_columns
                ),
                "numeric_columns": list(
                    numeric_columns
                ),
                "categorical_columns": list(
                    categorical_columns
                ),
                "target_column": target,
                "provenance_column": provenance,
                "identifier_columns": list(
                    identifiers
                ),
            },

            "leakage_policy": {
                "target_used_as_predictor": False,
                "provenance_used": False,
                "identifiers_used": False,
            },

            "sampling": {
                "policy": SYNTHETIC_SAMPLE_POLICY,
                "training_rows": len(
                    TRAINING_DATA[
                        dataset_id
                    ]
                ),
                "synthetic_rows": len(
                    synthetic_df
                ),
            },

            "reproducibility": {
                "master_seed": MASTER_SEED,
                "baseline_seed": seed,
            },

            "runtime": runtime_records,

            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }

        # ------------------------------------------------------------------------------------------
        # Metadata Output Path
        # ------------------------------------------------------------------------------------------

        metadata_path = (
            NB04_METADATA_ROOT
            / dataset_id
            / f"{baseline_name}.json"
        )

        metadata_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        # ------------------------------------------------------------------------------------------
        # Save Metadata
        # ------------------------------------------------------------------------------------------

        with open(
            metadata_path,
            "w",
            encoding="utf-8",
        ) as handle:

            json.dump(
                metadata,
                handle,
                indent=2,
                default=str,
            )

        # ------------------------------------------------------------------------------------------
        # Validate Metadata Persistence
        # ------------------------------------------------------------------------------------------

        if not metadata_path.exists():

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "metadata file was not created."
            )

        if metadata_path.stat().st_size <= 0:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "metadata file is empty."
            )

        # ------------------------------------------------------------------------------------------
        # Reload Metadata
        # ------------------------------------------------------------------------------------------

        with open(
            metadata_path,
            "r",
            encoding="utf-8",
        ) as handle:

            reloaded_metadata = json.load(
                handle
            )

        if reloaded_metadata[
            "dataset_id"
        ] != dataset_id:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "persisted metadata dataset_id mismatch."
            )

        if reloaded_metadata[
            "baseline"
        ]["id"] != baseline_name:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "persisted metadata baseline mismatch."
            )

        # ------------------------------------------------------------------------------------------
        # Record Metadata Artifact
        # ------------------------------------------------------------------------------------------

        BASELINE_METADATA_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "metadata_path": str(
                    metadata_path
                ),
                "metadata_relative_path": str(
                    metadata_path.relative_to(
                        NB04_ROOT
                    )
                ),
                "metadata_size_bytes": (
                    metadata_path.stat().st_size
                ),
                "metadata_sha256": calculate_sha256(
                    metadata_path
                ),
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"metadata saved and verified"
        )

# --------------------------------------------------------------------------------------------------
# 3. Create Metadata Registry DataFrame
# --------------------------------------------------------------------------------------------------

BASELINE_METADATA_DF = pd.DataFrame(
    BASELINE_METADATA_RECORDS
)

EXPECTED_METADATA_RECORDS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

if len(
    BASELINE_METADATA_DF
) != EXPECTED_METADATA_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_METADATA_RECORDS} metadata records, "
        f"found {len(BASELINE_METADATA_DF)}."
    )

# --------------------------------------------------------------------------------------------------
# 4. Validate Metadata Registry Uniqueness
# --------------------------------------------------------------------------------------------------

if BASELINE_METADATA_DF.duplicated(
    subset=[
        "dataset_id",
        "baseline",
    ]
).any():

    raise RuntimeError(
        "Duplicate baseline metadata records detected."
    )

# --------------------------------------------------------------------------------------------------
# 5. Validate Metadata Coverage
# --------------------------------------------------------------------------------------------------

expected_metadata_keys = {
    (
        dataset_id,
        baseline_name,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}

actual_metadata_keys = {
    (
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in BASELINE_METADATA_DF.iterrows()
}

if actual_metadata_keys != expected_metadata_keys:

    raise RuntimeError(
        "Baseline metadata coverage mismatch."
    )

# --------------------------------------------------------------------------------------------------
# 6. Save Runtime Table
# --------------------------------------------------------------------------------------------------

RUNTIME_PATH = (
    NB04_METADATA_ROOT
    / "baseline_runtime.csv"
)

RUNTIME_DF.to_csv(
    RUNTIME_PATH,
    index=False,
)

if not RUNTIME_PATH.exists():

    raise RuntimeError(
        "Baseline runtime CSV was not created."
    )

if RUNTIME_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "Baseline runtime CSV is empty."
    )

# --------------------------------------------------------------------------------------------------
# 7. Final Integrity Summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Metadata records            : "
    f"{len(BASELINE_METADATA_DF)}"
)

print(
    "✓ Metadata persistence        : PASS"
)

print(
    "✓ Metadata reload validation  : PASS"
)

print(
    "✓ Metadata coverage           : PASS"
)

print(
    "✓ Metadata uniqueness         : PASS"
)

print(
    f"✓ Runtime table saved         : "
    f"{RUNTIME_PATH}"
)

print(
    "✓ SECTION 14 — BASELINE METADATA: PASS"
)

SECTION 14 — SAVE BASELINE METADATA
✓ adult_income         | independent_marginal     | metadata saved and verified
✓ adult_income         | gaussian_copula          | metadata saved and verified
✓ bank_marketing       | independent_marginal     | metadata saved and verified
✓ bank_marketing       | gaussian_copula          | metadata saved and verified
✓ diabetes_130us       | independent_marginal     | metadata saved and verified
✓ diabetes_130us       | gaussian_copula          | metadata saved and verified

----------------------------------------------------------------------------------------------------
✓ Metadata records            : 6
✓ Metadata persistence        : PASS
✓ Metadata reload validation  : PASS
✓ Metadata coverage           : PASS
✓ Metadata uniqueness         : PASS
✓ Runtime table saved         : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/metadata/baseline_runtime.csv
✓ SECTION 14 — BASELINE METADATA: PASS


In [53]:
# ==================================================================================================
# 15. SAVE GENERATION MANIFEST
# ==================================================================================================

print("=" * 100)
print("SECTION 15 — SAVE GENERATION MANIFEST")
print("=" * 100)

import joblib
from datetime import datetime, timezone

# --------------------------------------------------------------------------------------------------
# 1. Initialize Manifest Registry
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_RECORDS = []

EXPECTED_MANIFEST_RECORDS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

# --------------------------------------------------------------------------------------------------
# 2. Process Every Dataset × Baseline
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    # Authoritative training-row reference
    training_rows = len(
        TRAINING_DATA[
            dataset_id
        ]
    )

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        # ------------------------------------------------------------------------------------------
        # Validate Baseline Model Availability
        # ------------------------------------------------------------------------------------------

        if dataset_id not in FITTED_BASELINE_MODELS:

            raise RuntimeError(
                f"{dataset_id}: fitted baseline model container is missing."
            )

        if baseline_name not in FITTED_BASELINE_MODELS[
            dataset_id
        ]:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: fitted model is missing."
            )

        # ------------------------------------------------------------------------------------------
        # Model Path
        # ------------------------------------------------------------------------------------------

        model_directory = (
            NB04_MODEL_ROOT
            / dataset_id
        )

        model_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        model_path = (
            model_directory
            / f"{baseline_name}.pkl"
        )

        # ------------------------------------------------------------------------------------------
        # Save Fitted Model
        # ------------------------------------------------------------------------------------------

        fitted_model = (
            FITTED_BASELINE_MODELS[
                dataset_id
            ][baseline_name]
        )

        if baseline_name == "independent_marginal":

            joblib.dump(
                fitted_model,
                model_path,
                compress=3,
            )

        elif baseline_name == "gaussian_copula":

            if not isinstance(
                fitted_model,
                dict
            ):

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}: "
                    "Gaussian Copula fitted model container is invalid."
                )

            if "synthesizer" not in fitted_model:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}: "
                    "Gaussian Copula synthesizer is missing."
                )

            fitted_model[
                "synthesizer"
            ].save(
                filepath=str(
                    model_path
                )
            )

        else:

            raise ValueError(
                f"Unknown baseline: {baseline_name}"
            )

        # ------------------------------------------------------------------------------------------
        # Validate Persisted Model Artifact
        # ------------------------------------------------------------------------------------------

        if not model_path.exists():

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "model artifact was not created."
            )

        model_file_size = (
            model_path.stat().st_size
        )

        if model_file_size <= 0:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "model artifact is empty."
            )

        model_sha256 = calculate_sha256(
            model_path
        )

        if not model_sha256:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "model SHA-256 calculation failed."
            )

        # ------------------------------------------------------------------------------------------
        # Synthetic Artifact
        # ------------------------------------------------------------------------------------------

        synthetic_matches = (
            SYNTHETIC_ARTIFACT_DF[
                (
                    SYNTHETIC_ARTIFACT_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    SYNTHETIC_ARTIFACT_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ]
        )

        if len(
            synthetic_matches
        ) != 1:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"expected exactly one synthetic artifact record, "
                f"found {len(synthetic_matches)}."
            )

        synthetic_record = (
            synthetic_matches.iloc[0]
        )

        # ------------------------------------------------------------------------------------------
        # Metadata Artifact
        # ------------------------------------------------------------------------------------------

        metadata_matches = (
            BASELINE_METADATA_DF[
                (
                    BASELINE_METADATA_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    BASELINE_METADATA_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ]
        )

        if len(
            metadata_matches
        ) != 1:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"expected exactly one metadata artifact record, "
                f"found {len(metadata_matches)}."
            )

        metadata_record = (
            metadata_matches.iloc[0]
        )

        # ------------------------------------------------------------------------------------------
        # Runtime Artifact
        # ------------------------------------------------------------------------------------------

        runtime_matches = (
            RUNTIME_DF[
                (
                    RUNTIME_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    RUNTIME_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ]
        )

        if len(
            runtime_matches
        ) != 1:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"expected exactly one runtime record, "
                f"found {len(runtime_matches)}."
            )

        runtime_record = (
            runtime_matches.iloc[0]
        )

        # ------------------------------------------------------------------------------------------
        # Validate Training/Synthetic Row Counts
        # ------------------------------------------------------------------------------------------

        synthetic_rows = int(
            synthetic_record[
                "rows"
            ]
        )

        if synthetic_rows != training_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"synthetic rows ({synthetic_rows}) do not match "
                f"training rows ({training_rows})."
            )

        # ------------------------------------------------------------------------------------------
        # Construct Manifest Record
        # ------------------------------------------------------------------------------------------

        GENERATION_MANIFEST_RECORDS.append(
            {

                "notebook_id": NOTEBOOK_ID,

                "notebook_version": NOTEBOOK_VERSION,

                "dataset_id": dataset_id,

                "baseline": baseline_name,

                "fit_split": "train",

                "validation_used_for_fit": False,

                "test_used_for_fit": False,

                "target_used_as_predictor": False,

                "identifier_used": False,

                "provenance_used": False,

                "training_rows": training_rows,

                "synthetic_rows": synthetic_rows,

                "synthetic_columns": int(
                    synthetic_record[
                        "columns"
                    ]
                ),

                "synthetic_data_path": (
                    synthetic_record[
                        "relative_path"
                    ]
                ),

                "synthetic_data_sha256": (
                    synthetic_record[
                        "sha256"
                    ]
                ),

                "synthetic_file_size_bytes": int(
                    synthetic_record[
                        "file_size_bytes"
                    ]
                ),

                "model_path": str(
                    model_path.relative_to(
                        NB04_ROOT
                    )
                ),

                "model_sha256": model_sha256,

                "model_file_size_bytes": model_file_size,

                "metadata_path": str(
                    Path(
                        metadata_record[
                            "metadata_path"
                        ]
                    ).relative_to(
                        NB04_ROOT
                    )
                ),

                "metadata_sha256": (
                    metadata_record[
                        "metadata_sha256"
                    ]
                ),

                "seed": int(
                    runtime_record[
                        "seed"
                    ]
                ),

                "fit_runtime_seconds": float(
                    runtime_record[
                        "fit_runtime_seconds"
                    ]
                ),

                "generation_runtime_seconds": float(
                    runtime_record[
                        "generation_runtime_seconds"
                    ]
                ),

                "total_runtime_seconds": float(
                    runtime_record[
                        "total_runtime_seconds"
                    ]
                ),

                "status": "PASS",

                "created_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"model + manifest record prepared"
        )

# --------------------------------------------------------------------------------------------------
# 3. Create Manifest DataFrame
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_DF = pd.DataFrame(
    GENERATION_MANIFEST_RECORDS
)

# --------------------------------------------------------------------------------------------------
# 4. Validate Manifest Record Count
# --------------------------------------------------------------------------------------------------

if len(
    GENERATION_MANIFEST_DF
) != EXPECTED_MANIFEST_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_MANIFEST_RECORDS} manifest records, "
        f"found {len(GENERATION_MANIFEST_DF)}."
    )

# --------------------------------------------------------------------------------------------------
# 5. Validate Manifest Uniqueness
# --------------------------------------------------------------------------------------------------

manifest_keys = [
    "dataset_id",
    "baseline",
]

if GENERATION_MANIFEST_DF.duplicated(
    subset=manifest_keys
).any():

    raise RuntimeError(
        "Duplicate dataset × baseline manifest records detected."
    )

# --------------------------------------------------------------------------------------------------
# 6. Validate Manifest Coverage
# --------------------------------------------------------------------------------------------------

expected_manifest_keys = {
    (
        dataset_id,
        baseline_name,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}

actual_manifest_keys = {
    (
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_MANIFEST_DF.iterrows()
}

if actual_manifest_keys != expected_manifest_keys:

    missing_keys = (
        expected_manifest_keys
        - actual_manifest_keys
    )

    unexpected_keys = (
        actual_manifest_keys
        - expected_manifest_keys
    )

    raise RuntimeError(
        "Generation manifest coverage mismatch.\n"
        f"Missing: {sorted(missing_keys)}\n"
        f"Unexpected: {sorted(unexpected_keys)}"
    )

# --------------------------------------------------------------------------------------------------
# 7. Save Generation Manifest
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_PATH = (
    NB04_MANIFEST_ROOT
    / "generation_manifest.csv"
)

GENERATION_MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

GENERATION_MANIFEST_DF.to_csv(
    GENERATION_MANIFEST_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 8. Validate Manifest Persistence
# --------------------------------------------------------------------------------------------------

if not GENERATION_MANIFEST_PATH.exists():

    raise RuntimeError(
        "Generation manifest file was not created."
    )

if GENERATION_MANIFEST_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "Generation manifest file is empty."
    )

# Reload persisted manifest
RELOADED_GENERATION_MANIFEST_DF = pd.read_csv(
    GENERATION_MANIFEST_PATH
)

if len(
    RELOADED_GENERATION_MANIFEST_DF
) != EXPECTED_MANIFEST_RECORDS:

    raise RuntimeError(
        "Persisted generation manifest record count mismatch."
    )

if set(
    RELOADED_GENERATION_MANIFEST_DF[
        "dataset_id"
    ]
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Persisted generation manifest dataset coverage mismatch."
    )

if set(
    RELOADED_GENERATION_MANIFEST_DF[
        "baseline"
    ]
) != set(
    BASELINE_METHODS
):

    raise RuntimeError(
        "Persisted generation manifest baseline coverage mismatch."
    )

# --------------------------------------------------------------------------------------------------
# 9. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Manifest records             : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print(
    "✓ Model artifact persistence   : PASS"
)

print(
    "✓ Model SHA-256 recording      : PASS"
)

print(
    "✓ Training-row consistency     : PASS"
)

print(
    "✓ Synthetic-row consistency    : PASS"
)

print(
    "✓ Manifest coverage            : PASS"
)

print(
    "✓ Manifest uniqueness          : PASS"
)

print(
    "✓ Manifest persistence         : PASS"
)

print(
    "✓ Manifest reload validation   : PASS"
)

print(
    f"✓ Generation manifest saved    : "
    f"{GENERATION_MANIFEST_PATH}"
)

print(
    "✓ SECTION 15 — GENERATION MANIFEST: PASS"
)

SECTION 15 — SAVE GENERATION MANIFEST
✓ adult_income         | independent_marginal     | model + manifest record prepared
✓ adult_income         | gaussian_copula          | model + manifest record prepared
✓ bank_marketing       | independent_marginal     | model + manifest record prepared
✓ bank_marketing       | gaussian_copula          | model + manifest record prepared
✓ diabetes_130us       | independent_marginal     | model + manifest record prepared
✓ diabetes_130us       | gaussian_copula          | model + manifest record prepared

----------------------------------------------------------------------------------------------------
✓ Manifest records             : 6
✓ Model artifact persistence   : PASS
✓ Model SHA-256 recording      : PASS
✓ Training-row consistency     : PASS
✓ Synthetic-row consistency    : PASS
✓ Manifest coverage            : PASS
✓ Manifest uniqueness          : PASS
✓ Manifest persistence         : PASS
✓ Manifest reload validation   : PASS
✓ Generatio

In [55]:
# ==================================================================================================
# 16. VERIFY ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("SECTION 16 — VERIFY ARTIFACTS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_RUNS = (
    len(DATASET_IDS)
    *
    len(BASELINE_METHODS)
)

if len(
    GENERATION_MANIFEST_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Generation manifest count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {len(GENERATION_MANIFEST_DF)}"
    )

# --------------------------------------------------------------------------------------------------
# 2. Verification Containers
# --------------------------------------------------------------------------------------------------

ARTIFACT_VERIFICATION_RECORDS = []

# --------------------------------------------------------------------------------------------------
# 3. Verify Every Dataset × Baseline
# --------------------------------------------------------------------------------------------------

for _, record in GENERATION_MANIFEST_DF.iterrows():

    dataset_id = record[
        "dataset_id"
    ]

    baseline_name = record[
        "baseline"
    ]

    # ----------------------------------------------------------------------------------------------
    # Paths
    # ----------------------------------------------------------------------------------------------

    synthetic_path = (
        NB04_ROOT
        / record[
            "synthetic_data_path"
        ]
    )

    model_path = (
        NB04_ROOT
        / record[
            "model_path"
        ]
    )

    metadata_path = (
        NB04_ROOT
        / record[
            "metadata_path"
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Existence
    # ----------------------------------------------------------------------------------------------

    for label, path in [
        ("synthetic data", synthetic_path),
        ("model", model_path),
        ("metadata", metadata_path),
    ]:

        if not path.exists():

            raise FileNotFoundError(
                f"{dataset_id}/{baseline_name}: "
                f"{label} artifact missing:\n{path}"
            )

        if path.stat().st_size <= 0:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"{label} artifact is empty:\n{path}"
            )

    # ----------------------------------------------------------------------------------------------
    # Synthetic Hash Verification
    # ----------------------------------------------------------------------------------------------

    actual_synthetic_hash = calculate_sha256(
        synthetic_path
    )

    if actual_synthetic_hash != record[
        "synthetic_data_sha256"
    ]:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "synthetic-data SHA-256 mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Model Hash Verification
    # ----------------------------------------------------------------------------------------------

    actual_model_hash = calculate_sha256(
        model_path
    )

    if actual_model_hash != record[
        "model_sha256"
    ]:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "model SHA-256 mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Metadata Hash Verification
    # ----------------------------------------------------------------------------------------------

    actual_metadata_hash = calculate_sha256(
        metadata_path
    )

    if actual_metadata_hash != record[
        "metadata_sha256"
    ]:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "metadata SHA-256 mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Reload Persisted Synthetic Dataset
    # ----------------------------------------------------------------------------------------------

    reloaded_df = pd.read_csv(
        synthetic_path,
        low_memory=False,
    )

    # ----------------------------------------------------------------------------------------------
    # Expected Schema
    # ----------------------------------------------------------------------------------------------

    expected_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    if list(
        reloaded_df.columns
    ) != list(
        expected_columns
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "persisted synthetic schema mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Authoritative Training Row Count
    # ----------------------------------------------------------------------------------------------

    expected_rows = len(
        TRAINING_DATA[
            dataset_id
        ]
    )

    if len(
        reloaded_df
    ) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "persisted synthetic row count mismatch.\n"
            f"Expected: {expected_rows}\n"
            f"Found   : {len(reloaded_df)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Manifest Row Consistency
    # ----------------------------------------------------------------------------------------------

    if int(
        record[
            "synthetic_rows"
        ]
    ) != len(
        reloaded_df
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "manifest synthetic-row count mismatch."
        )

    if int(
        record[
            "synthetic_columns"
        ]
    ) != len(
        reloaded_df.columns
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "manifest synthetic-column count mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Verify Forbidden Columns
    # ----------------------------------------------------------------------------------------------

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    if provenance in reloaded_df.columns:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "persisted provenance leakage."
        )

    leaked_ids = (
        set(
            identifiers
        )
        .intersection(
            reloaded_df.columns
        )
    )

    if leaked_ids:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            f"persisted identifier leakage: {leaked_ids}"
        )

    # ----------------------------------------------------------------------------------------------
    # Verify Target Presence
    # ----------------------------------------------------------------------------------------------

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    if target not in reloaded_df.columns:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            f"target column '{target}' missing from synthetic data."
        )

    # ----------------------------------------------------------------------------------------------
    # Verify Manifest Research Controls
    # ----------------------------------------------------------------------------------------------

    if record[
        "fit_split"
    ] != "train":

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "manifest fit split is not 'train'."
        )

    if bool(
        record[
            "validation_used_for_fit"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "validation data was marked as used for fitting."
        )

    if bool(
        record[
            "test_used_for_fit"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "test data was marked as used for fitting."
        )

    if bool(
        record[
            "target_used_as_predictor"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "target leakage flag is active."
        )

    if bool(
        record[
            "identifier_used"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "identifier usage flag is active."
        )

    if bool(
        record[
            "provenance_used"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "provenance usage flag is active."
        )

    # ----------------------------------------------------------------------------------------------
    # Record Successful Verification
    # ----------------------------------------------------------------------------------------------

    ARTIFACT_VERIFICATION_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "baseline": baseline_name,
            "synthetic_data_verified": True,
            "model_verified": True,
            "metadata_verified": True,
            "schema_verified": True,
            "sample_size_verified": True,
            "target_separation_verified": True,
            "identifier_exclusion_verified": True,
            "provenance_exclusion_verified": True,
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"{baseline_name:<24} | "
        f"CSV PASS | MODEL PASS | METADATA PASS"
    )

# --------------------------------------------------------------------------------------------------
# 4. Create Verification DataFrame
# --------------------------------------------------------------------------------------------------

ARTIFACT_VERIFICATION_DF = pd.DataFrame(
    ARTIFACT_VERIFICATION_RECORDS
)

# --------------------------------------------------------------------------------------------------
# 5. Final Verification Count
# --------------------------------------------------------------------------------------------------

if len(
    ARTIFACT_VERIFICATION_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Artifact verification record count mismatch."
    )

# --------------------------------------------------------------------------------------------------
# 6. Verify Coverage and Uniqueness
# --------------------------------------------------------------------------------------------------

if ARTIFACT_VERIFICATION_DF.duplicated(
    subset=[
        "dataset_id",
        "baseline",
    ]
).any():

    raise RuntimeError(
        "Duplicate artifact verification records detected."
    )

expected_verification_keys = {
    (
        dataset_id,
        baseline_name,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}

actual_verification_keys = {
    (
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in ARTIFACT_VERIFICATION_DF.iterrows()
}

if actual_verification_keys != expected_verification_keys:

    raise RuntimeError(
        "Artifact verification coverage mismatch."
    )

# --------------------------------------------------------------------------------------------------
# 7. Save Artifact Validation Report
# --------------------------------------------------------------------------------------------------

ARTIFACT_VALIDATION_REPORT = {

    "notebook_id": NOTEBOOK_ID,

    "notebook_version": NOTEBOOK_VERSION,

    "expected_runs": EXPECTED_RUNS,

    "verified_runs": len(
        ARTIFACT_VERIFICATION_DF
    ),

    "datasets": list(
        DATASET_IDS
    ),

    "baselines": list(
        BASELINE_METHODS
    ),

    "all_synthetic_data_verified": bool(
        ARTIFACT_VERIFICATION_DF[
            "synthetic_data_verified"
        ].all()
    ),

    "all_models_verified": bool(
        ARTIFACT_VERIFICATION_DF[
            "model_verified"
        ].all()
    ),

    "all_metadata_verified": bool(
        ARTIFACT_VERIFICATION_DF[
            "metadata_verified"
        ].all()
    ),

    "schema_verification": bool(
        ARTIFACT_VERIFICATION_DF[
            "schema_verified"
        ].all()
    ),

    "sample_size_verification": bool(
        ARTIFACT_VERIFICATION_DF[
            "sample_size_verified"
        ].all()
    ),

    "target_separation_verification": bool(
        ARTIFACT_VERIFICATION_DF[
            "target_separation_verified"
        ].all()
    ),

    "identifier_exclusion_verification": bool(
        ARTIFACT_VERIFICATION_DF[
            "identifier_exclusion_verified"
        ].all()
    ),

    "provenance_exclusion_verification": bool(
        ARTIFACT_VERIFICATION_DF[
            "provenance_exclusion_verified"
        ].all()
    ),

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

ARTIFACT_VALIDATION_PATH = (
    NB04_VALIDATION_ROOT
    / "artifact_validation.json"
)

with open(
    ARTIFACT_VALIDATION_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        ARTIFACT_VALIDATION_REPORT,
        handle,
        indent=2,
    )

# --------------------------------------------------------------------------------------------------
# 8. Reload Validation Report
# --------------------------------------------------------------------------------------------------

if not ARTIFACT_VALIDATION_PATH.exists():

    raise RuntimeError(
        "Artifact validation report was not created."
    )

with open(
    ARTIFACT_VALIDATION_PATH,
    "r",
    encoding="utf-8",
) as handle:

    reloaded_validation_report = json.load(
        handle
    )

if reloaded_validation_report[
    "verified_runs"
] != EXPECTED_RUNS:

    raise RuntimeError(
        "Persisted artifact validation report "
        "contains an incorrect verified-run count."
    )

# --------------------------------------------------------------------------------------------------
# 9. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Expected runs                    : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Verified runs                    : "
    f"{len(ARTIFACT_VERIFICATION_DF)}"
)

print(
    "✓ Synthetic data verification     : PASS"
)

print(
    "✓ Model verification              : PASS"
)

print(
    "✓ Metadata verification           : PASS"
)

print(
    "✓ SHA-256 verification            : PASS"
)

print(
    "✓ Schema verification              : PASS"
)

print(
    "✓ Sample-size verification        : PASS"
)

print(
    "✓ Target separation verification  : PASS"
)

print(
    "✓ Identifier exclusion             : PASS"
)

print(
    "✓ Provenance exclusion             : PASS"
)

print(
    "✓ Artifact validation report saved:"
)

print(
    f"  {ARTIFACT_VALIDATION_PATH}"
)

print()
print(
    "✓ SECTION 16 — ARTIFACT VERIFICATION : PASS"
)

SECTION 16 — VERIFY ARTIFACTS
✓ adult_income         | independent_marginal     | CSV PASS | MODEL PASS | METADATA PASS
✓ adult_income         | gaussian_copula          | CSV PASS | MODEL PASS | METADATA PASS
✓ bank_marketing       | independent_marginal     | CSV PASS | MODEL PASS | METADATA PASS
✓ bank_marketing       | gaussian_copula          | CSV PASS | MODEL PASS | METADATA PASS
✓ diabetes_130us       | independent_marginal     | CSV PASS | MODEL PASS | METADATA PASS
✓ diabetes_130us       | gaussian_copula          | CSV PASS | MODEL PASS | METADATA PASS

----------------------------------------------------------------------------------------------------
✓ Expected runs                    : 6
✓ Verified runs                    : 6
✓ Synthetic data verification     : PASS
✓ Model verification              : PASS
✓ Metadata verification           : PASS
✓ SHA-256 verification            : PASS
✓ Schema verification              : PASS
✓ Sample-size verification        : PASS
✓ T

In [57]:
# ==================================================================================================
# 17. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("SECTION 17 — COMPLETION SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Notebook Summary
# --------------------------------------------------------------------------------------------------

print()
print("NOTEBOOK 04 — STATISTICAL BASELINES")
print("=" * 100)

print(
    f"Notebook version       : {NOTEBOOK_VERSION}"
)

print(
    f"Project root           : {PROJECT_ROOT}"
)

print(
    f"Notebook 02 input      : {CANONICAL_NB02_ROOT}"
)

print(
    f"Notebook 04 output     : {NB04_ROOT}"
)

# --------------------------------------------------------------------------------------------------
# 2. Dataset Summary
# --------------------------------------------------------------------------------------------------

print()
print("DATASETS")
print("-" * 100)

for dataset_id in DATASET_IDS:

    training_rows = len(
        TRAINING_DATA[
            dataset_id
        ]
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"training rows={training_rows:,}"
    )

# --------------------------------------------------------------------------------------------------
# 3. Baseline Summary
# --------------------------------------------------------------------------------------------------

print()
print("BASELINES")
print("-" * 100)

for baseline_name in BASELINE_METHODS:

    print(
        f"✓ {baseline_name:<24} | "
        f"{BASELINE_DEFINITIONS[baseline_name]['name']}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Experimental Integrity
# --------------------------------------------------------------------------------------------------

print()
print("EXPERIMENTAL INTEGRITY")
print("-" * 100)

print("✓ TRAIN split only used for fitting")
print("✓ VALIDATION split excluded from fitting")
print("✓ TEST split excluded from fitting")
print("✓ Notebook 02 preprocessing not refitted")
print("✓ Raw datasets not reloaded")
print("✓ Native generative schema used")
print("✓ Target retained in synthetic data")
print("✓ Target never used as predictor")
print("✓ Provenance excluded")
print("✓ Identifiers excluded")
print("✓ Synthetic size equals training size")
print("✓ Reproducible seed policy applied")

# --------------------------------------------------------------------------------------------------
# 5. Artifact Summary
# --------------------------------------------------------------------------------------------------

print()
print("ARTIFACTS")
print("-" * 100)

print(
    f"✓ Synthetic datasets : "
    f"{len(SYNTHETIC_ARTIFACT_DF)}"
)

print(
    f"✓ Model artifacts    : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print(
    f"✓ Metadata artifacts : "
    f"{len(BASELINE_METADATA_DF)}"
)

print(
    f"✓ Manifest records   : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print(
    f"✓ Verified runs      : "
    f"{len(ARTIFACT_VERIFICATION_DF)}"
)

# --------------------------------------------------------------------------------------------------
# 6. Final Artifact Integrity Gate
# --------------------------------------------------------------------------------------------------

EXPECTED_RUNS = (
    len(DATASET_IDS)
    *
    len(BASELINE_METHODS)
)

if len(
    SYNTHETIC_ARTIFACT_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Synthetic artifact count mismatch."
    )

if len(
    BASELINE_METADATA_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Baseline metadata count mismatch."
    )

if len(
    GENERATION_MANIFEST_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Generation manifest count mismatch."
    )

if len(
    ARTIFACT_VERIFICATION_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Artifact verification count mismatch."
    )

if not (
    ARTIFACT_VERIFICATION_DF[
        "status"
    ] == "PASS"
).all():

    raise RuntimeError(
        "One or more artifact verification records are not PASS."
    )

# --------------------------------------------------------------------------------------------------
# 7. Output Locations
# --------------------------------------------------------------------------------------------------

print()
print("OUTPUT LOCATIONS")
print("-" * 100)

print(
    f"Synthetic : {NB04_SYNTHETIC_ROOT}"
)

print(
    f"Models    : {NB04_MODEL_ROOT}"
)

print(
    f"Metadata  : {NB04_METADATA_ROOT}"
)

print(
    f"Manifests : {NB04_MANIFEST_ROOT}"
)

print(
    f"Validation: {NB04_VALIDATION_ROOT}"
)

# --------------------------------------------------------------------------------------------------
# 8. Final Completion Status
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Expected experimental runs : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Synthetic artifacts        : "
    f"{len(SYNTHETIC_ARTIFACT_DF)}"
)

print(
    f"✓ Model artifacts            : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print(
    f"✓ Metadata artifacts         : "
    f"{len(BASELINE_METADATA_DF)}"
)

print(
    f"✓ Manifest records           : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print(
    f"✓ Artifact verification     : "
    f"{len(ARTIFACT_VERIFICATION_DF)}/{EXPECTED_RUNS} PASS"
)

print()
print("=" * 100)
print("✓ NOTEBOOK 04 — STATISTICAL BASELINES : PASS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 9. RAM Cleanup
# --------------------------------------------------------------------------------------------------

gc.collect()

print()
print("✓ RAM cleanup completed.")

SECTION 17 — COMPLETION SUMMARY

NOTEBOOK 04 — STATISTICAL BASELINES
Notebook version       : 2.0
Project root           : /content/drive/MyDrive/SPP_GAN_Research
Notebook 02 input      : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
Notebook 04 output     : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04

DATASETS
----------------------------------------------------------------------------------------------------
✓ adult_income         | training rows=34,189
✓ bank_marketing       | training rows=31,647
✓ diabetes_130us       | training rows=71,236

BASELINES
----------------------------------------------------------------------------------------------------
✓ independent_marginal     | Independent Marginal Sampling
✓ gaussian_copula          | Gaussian Copula

EXPERIMENTAL INTEGRITY
----------------------------------------------------------------------------------------------------
✓ TRAIN split only used for fitting
✓ VALIDATION split excluded